In [1]:

!git clone https://github.com/Shruti-sivaramakrishnan/indic-fake-voice-detection.git


Cloning into 'indic-fake-voice-detection'...
remote: Enumerating objects: 357, done.
remote: Counting objects: 100% (92/92), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 357 (delta 36), reused 68 (delta 21), pack-reused 265 (from 1)
Receiving objects: 100% (357/357), 6.79 MiB | 15.65 MiB/s, done.
Resolving deltas: 100% (145/145), done.


In [2]:

%cd /content/indic-fake-voice-detection


/content/indic-fake-voice-detection


In [3]:

!find . -maxdepth 2 -type f | sort



./app.py
./checkpoints/best_model.pt
./checkpoints/per_language_eer.json
./checkpoints/step6_summary.json
./codec_robustness.py
./confusion_matrix_Assamese.png
./confusion_matrix_Bengali.png
./confusion_matrix_Bodo.png
./confusion_matrix_Dogri.png
./confusion_matrix_English.png
./confusion_matrix_Gujarati.png
./confusion_matrix_Hindi.png
./confusion_matrix_Kannada.png
./confusion_matrix_Malayalam.png
./confusion_matrix_Manipuri.png
./confusion_matrix_Marathi.png
./confusion_matrix_Nepali.png
./confusion_matrix_Odia.png
./confusion_matrix_Sanskrit.png
./confusion_matrix_Tamil.png
./confusion_matrix_Telugu.png
./data_pipeline/build_group_safe_split.py
./data_pipeline/export_hf_metadata.py
./data_pipeline/.gitkeep
./data_pipeline/leakage_audit.py
./data_pipeline/split_ids.json
./data_pipeline/split_protocol.py
./docs/COLAB_STEP6.md
./docs/KAGGLE_DEPLOYMENT.md
./docs/TEAM_SYNC_LOG.md
./docs/TEST_SET_RESULTS.md
./docs/V2_TRAINING.md
./docs/Voice_Guard_Project_Report.docx
./evaluate_test_spl

In [4]:

!grep -n -i -E "30|duration|max_seconds|max_duration|AudioInputPolicy" voice_guard_core.py model_pipeline.py app.py


voice_guard_core.py:42:class AudioInputPolicy:
voice_guard_core.py:46:    min_duration_seconds: float = 1.0
voice_guard_core.py:47:    max_duration_seconds: float = 30.0
voice_guard_core.py:70:DEFAULT_AUDIO_POLICY = AudioInputPolicy()
voice_guard_core.py:76:    policy: AudioInputPolicy = DEFAULT_AUDIO_POLICY,
voice_guard_core.py:92:    policy: AudioInputPolicy = DEFAULT_AUDIO_POLICY,
voice_guard_core.py:104:    duration = samples.size / float(sample_rate)
voice_guard_core.py:105:    if duration < policy.min_duration_seconds:
voice_guard_core.py:107:            f"The clip is too short. Use at least {policy.min_duration_seconds:.0f} second "
voice_guard_core.py:110:    if duration > policy.max_duration_seconds:
voice_guard_core.py:112:            f"The clip is too long. Use a clip of {policy.max_duration_seconds:.0f} seconds "
voice_guard_core.py:150:        "duration_seconds": round(duration, 3),
model_pipeline.py:5:    frozen wav2vec2-XLS-R-300M frontend  ->  AASIST-inspired graph-atte

In [5]:

with open("voice_guard_core.py", "r") as f:
    lines = f.readlines()

for i in range(35, 120):
    print(f"{i+1}: {lines[i]}", end="")


36: 
37: class AudioValidationError(ValueError):
38:     """A safe, user-displayable rejection of unsuitable input audio."""
39: 
40: 
41: @dataclass(frozen=True)
42: class AudioInputPolicy:
43:     """Limits that protect both result quality and quadratic GAT inference."""
44: 
45:     max_bytes: int = 25 * 1024 * 1024
46:     min_duration_seconds: float = 1.0
47:     max_duration_seconds: float = 30.0
48:     min_rms: float = 1e-4
49:     min_peak: float = 1e-3
50:     min_active_seconds: float = 0.45
51: 
52: 
53: @dataclass(frozen=True)
54: class DecisionPolicy:
55:     """Three-way abstention policy operating on calibrated P(fake)."""
56: 
57:     policy_id: str = "legacy-symmetric-abstention-v1"
58:     real_max_p_fake: float = 0.35
59:     fake_min_p_fake: float = 0.65
60:     provenance: str = (
61:         "Legacy symmetric abstention band; not selected on the frozen test set. "
62:         "Replace only from validation data under the v2 protocol."
63:     )
64: 
65:     def __

In [6]:

with open("voice_guard_core.py", "r") as f:
    lines = f.readlines()

for i in range(35, 120):
    print(f"{i+1}: {lines[i]}", end="")


36: 
37: class AudioValidationError(ValueError):
38:     """A safe, user-displayable rejection of unsuitable input audio."""
39: 
40: 
41: @dataclass(frozen=True)
42: class AudioInputPolicy:
43:     """Limits that protect both result quality and quadratic GAT inference."""
44: 
45:     max_bytes: int = 25 * 1024 * 1024
46:     min_duration_seconds: float = 1.0
47:     max_duration_seconds: float = 30.0
48:     min_rms: float = 1e-4
49:     min_peak: float = 1e-3
50:     min_active_seconds: float = 0.45
51: 
52: 
53: @dataclass(frozen=True)
54: class DecisionPolicy:
55:     """Three-way abstention policy operating on calibrated P(fake)."""
56: 
57:     policy_id: str = "legacy-symmetric-abstention-v1"
58:     real_max_p_fake: float = 0.35
59:     fake_min_p_fake: float = 0.65
60:     provenance: str = (
61:         "Legacy symmetric abstention band; not selected on the frozen test set. "
62:         "Replace only from validation data under the v2 protocol."
63:     )
64: 
65:     def __

In [7]:

for i in range(115, 165):
    print(f"{i+1}: {lines[i]}", end="")


116:     peak = float(np.max(np.abs(samples)))
117:     rms = float(np.sqrt(np.mean(np.square(samples, dtype=np.float64))))
118:     if peak < policy.min_peak or rms < policy.min_rms:
119:         raise AudioValidationError(
120:             "No clear speech was detected. Upload a recording with an audible voice."
121:         )
122: 
123:     # A lightweight energy gate catches digital silence and clips with only a
124:     # short click. It is intentionally conservative: this is not a language- or
125:     # speaker-dependent VAD and should not reject quiet but usable speech.
126:     frame_size = max(1, int(round(sample_rate * 0.025)))
127:     hop_size = max(1, int(round(sample_rate * 0.010)))
128:     if samples.size <= frame_size:
129:         frame_starts = [0]
130:     else:
131:         frame_starts = range(0, samples.size - frame_size + 1, hop_size)
132:     frame_rms = np.asarray(
133:         [
134:             np.sqrt(np.mean(np.square(samples[start : start + frame_size], 

In [9]:

!pip install pypdf


In [11]:
!pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 16.6 MB/s  0:00:00


In [12]:
!pytest -q

.......................                                                  [100%]
23 passed in 2.14s


In [13]:
!git branch --show-current && git status --short

main


In [14]:
!git switch -c feature/long-call-analysis

Switched to a new branch 'feature/long-call-analysis'


In [15]:
!grep -n -E "^def (decode_audio_bytes|load_inference_bundle|score_waveform)|^class " model_pipeline.py

117:class TrainConfig:
1860:class ShardedFeatureDataset(Dataset):
1941:class ShardBlockShuffleSampler(Sampler):
2014:class GraphAttentionLayer(nn.Module):
2055:class AASISTHead(nn.Module):
2120:class PoolingBaselineHead(nn.Module):
2189:class CachedFeatureDataset(Dataset):
2304:class TemperatureScaler(nn.Module):
2822:class AnalysisResult:
2882:def decode_audio_bytes(audio_bytes: bytes) -> Tuple[np.ndarray, int]:
2936:class InferenceBundle:
2955:def load_inference_bundle(
3037:def score_waveform(


In [16]:
!sed -n '2870,3045p' model_pipeline.py


    return verdict_from_probability(
        p_fake,
        DecisionPolicy(
            policy_id="caller-supplied",
            real_max_p_fake=real_max,
            fake_min_p_fake=fake_min,
            provenance="Caller-supplied thresholds.",
        ),
    )


def decode_audio_bytes(audio_bytes: bytes) -> Tuple[np.ndarray, int]:
    """
    Decode arbitrary audio bytes (wav/flac/mp3/ogg/m4a) to a mono float32 array.

    Tries soundfile, then torchaudio (with ffmpeg backend), then pydub.
    This is also used by decode_audio() for encoded dataset examples.
    """
    import io

    last_error: Optional[BaseException] = None

    try:
        import soundfile as sf

        data, sr = sf.read(io.BytesIO(audio_bytes), dtype="float32", always_2d=False)
        if getattr(data, "ndim", 1) > 1:
            data = data.mean(axis=1)          # downmix to mono
        return np.asarray(data, dtype=np.float32), int(sr)
    except Exception as exc:  # noqa: BLE001
        last_error = ex

In [17]:
!sed -n '3037,3125p' model_pipeline.py

def score_waveform(
    waveform: np.ndarray,
    sample_rate: int = TARGET_SAMPLE_RATE,
    bundle: Optional[InferenceBundle] = None,
) -> Tuple[float, float]:
    """
    Score one waveform. Returns (raw_logit, calibrated_p_fake).

    Higher logit = more fake, matching the training target (1 = fake).
    """
    bundle = bundle or load_inference_bundle()

    feats = extract_features(waveform, sampling_rate=sample_rate)
    if feats.shape[0] == 0:
        raise ValueError("Frontend produced zero frames — clip is too short.")

    x = feats.unsqueeze(0).float().to(DEVICE)
    mask = torch.ones(1, feats.shape[0], device=DEVICE)

    logit = bundle.head(x, mask=mask)
    raw_logit = float(logit.reshape(-1)[0].item())

    calibrated = raw_logit / bundle.temperature
    p_fake = stable_sigmoid(calibrated)

    return raw_logit, p_fake


def analyze_bytes(
    audio_bytes: bytes,
    language_hint: Optional[str] = None,
    checkpoint_dir: str = CHECKPOINT_DIR,
) -> AnalysisResult:
    "

In [18]:
!sed -n '1,165p' voice_guard_core.py

"""Dependency-light safety and decision helpers for Voice Guard.

This module deliberately does not import torch or Streamlit.  Keeping the input
policy and result policy here makes them cheap to unit-test and lets the web UI
validate an upload before the multi-gigabyte frontend is loaded.
"""

from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Any, Dict

import numpy as np


SUPPORTED_LANGUAGES = (
    "Assamese",
    "Bengali",
    "Bodo",
    "Dogri",
    "English",
    "Gujarati",
    "Hindi",
    "Kannada",
    "Malayalam",
    "Manipuri",
    "Marathi",
    "Nepali",
    "Odia",
    "Sanskrit",
    "Tamil",
    "Telugu",
)


class AudioValidationError(ValueError):
    """A safe, user-displayable rejection of unsuitable input audio."""


@dataclass(frozen=True)
class AudioInputPolicy:
    """Limits that protect both result quality and quadratic GAT inference."""

    max_bytes: int = 25 * 1024 * 1024
    min_duration_seconds: float

In [19]:
!sed -n '1,240p' tests/test_voice_guard_core.py

import math

import numpy as np
import pytest

from voice_guard_core import (
    AudioInputPolicy,
    AudioValidationError,
    normalize_language_hint,
    stable_sigmoid,
    validate_decoded_audio,
    validate_encoded_audio_size,
    verdict_from_probability,
)


def test_encoded_audio_rejects_empty_and_oversized() -> None:
    with pytest.raises(AudioValidationError, match="empty"):
        validate_encoded_audio_size(b"")
    with pytest.raises(AudioValidationError, match="too large"):
        validate_encoded_audio_size(b"12345", AudioInputPolicy(max_bytes=4))


def test_decoded_audio_rejects_silence_before_inference() -> None:
    silence = np.zeros(16000, dtype=np.float32)
    with pytest.raises(AudioValidationError, match="No clear speech"):
        validate_decoded_audio(silence, 16000)


def test_decoded_audio_rejects_click_with_no_sustained_speech() -> None:
    waveform = np.zeros(32000, dtype=np.float32)
    waveform[1000:1010] = 0.8
    with pytest.raises(AudioValidat

In [20]:
!grep -R -n -E "resample|TARGET_SAMPLE_RATE|silence|speech.region|active_frames" --include="*.py" .

./tests/test_voice_guard_core.py:24:def test_decoded_audio_rejects_silence_before_inference() -> None:
./tests/test_voice_guard_core.py:25:    silence = np.zeros(16000, dtype=np.float32)
./tests/test_voice_guard_core.py:27:        validate_decoded_audio(silence, 16000)
./voice_guard_core.py:123:    # A lightweight energy gate catches digital silence and clips with only a
./voice_guard_core.py:140:    active_frames = int(np.count_nonzero(frame_rms >= active_threshold))
./voice_guard_core.py:141:    active_seconds = active_frames * hop_size / float(sample_rate)
./test_analyze_bytes_e2e.py:118:        ("digital silence", encode_wav_bytes(__import__("numpy").zeros(16000), 16000)),
./model_pipeline.py:102:TARGET_SAMPLE_RATE = 16000
./model_pipeline.py:463:    target_sample_rate: int = TARGET_SAMPLE_RATE,
./model_pipeline.py:474:    resampled = torchaudio.functional.resample(
./model_pipeline.py:479:    return resampled.numpy()
./model_pipeline.py:536:    sampling_rate: int = TARGET_SAMPLE_R

In [21]:
!sed -n '440,490p' model_pipeline.py

    training works without torchcodec.  The legacy AudioDecoder path remains
    supported for callers holding examples created by older datasets versions.
    """
    audio = example["audio"]
    if isinstance(audio, dict):
        encoded = audio.get("bytes")
        if encoded is not None:
            return decode_audio_bytes(encoded)

        path = audio.get("path")
        if path:
            with open(path, "rb") as handle:
                return decode_audio_bytes(handle.read())

        raise ValueError("Dataset audio has neither encoded bytes nor a path.")

    decoded = audio.get_all_samples()
    return decoded.data.numpy().squeeze(), decoded.sample_rate


def ensure_target_sample_rate(
    waveform: np.ndarray,
    sample_rate: int,
    target_sample_rate: int = TARGET_SAMPLE_RATE,
) -> np.ndarray:
    """
    Guard against a silent correctness bug: XLS-R expects 16 kHz, and feeding it
    another rate degrades features without raising anything.
    """
    if sample_rat

In [22]:
!cat requirements.txt

# Runtime dependencies for app.py and analyze_bytes(). Upper bounds prevent a
# new major release from silently changing the public Kaggle demo. Torch and
# torchaudio must resolve to the same release family.
streamlit>=1.40,<2
torch>=2.2,<3
torchaudio>=2.2,<3
transformers>=4.40,<6
numpy>=1.26,<3
soundfile>=0.12,<1
pydub>=0.25,<1
reportlab>=4.0,<5
truststore>=0.9,<1


In [23]:
!ls -la

total 1192
drwxr-xr-x 14 root root   4096 Nov 16 12:31  .
drwxr-xr-x  1 root root   4096 Nov 16 12:29  ..
-rw-r--r--  1 root root  18431 Nov 16 12:29  app.py
drwxr-xr-x  2 root root   4096 Nov 16 12:29  checkpoints
-rw-r--r--  1 root root   9352 Nov 16 12:29  codec_robustness.py
-rw-r--r--  1 root root  51769 Nov 16 12:29  confusion_matrix_Assamese.png
-rw-r--r--  1 root root  49111 Nov 16 12:29  confusion_matrix_Bengali.png
-rw-r--r--  1 root root  47348 Nov 16 12:29  confusion_matrix_Bodo.png
-rw-r--r--  1 root root  48572 Nov 16 12:29  confusion_matrix_Dogri.png
-rw-r--r--  1 root root  47031 Nov 16 12:29  confusion_matrix_English.png
-rw-r--r--  1 root root  48561 Nov 16 12:29  confusion_matrix_Gujarati.png
-rw-r--r--  1 root root  46931 Nov 16 12:29  confusion_matrix_Hindi.png
-rw-r--r--  1 root root  49678 Nov 16 12:29  confusion_matrix_Kannada.png
-rw-r--r--  1 root root  49806 Nov 16 12:29  confusion_matrix_Malayalam.png
-rw-r--r--  1 root root  48201 Nov 16 12:29  confusion_ma

In [24]:
!ls tests

__pycache__		  test_model_pipeline_audio.py
test_codec_robustness.py  test_split_protocol.py
test_evidence_report.py   test_voice_guard_core.py


In [25]:
!sed -n '30,75p' voice_guard_core.py

    "Odia",
    "Sanskrit",
    "Tamil",
    "Telugu",
)


class AudioValidationError(ValueError):
    """A safe, user-displayable rejection of unsuitable input audio."""


@dataclass(frozen=True)
class AudioInputPolicy:
    """Limits that protect both result quality and quadratic GAT inference."""

    max_bytes: int = 25 * 1024 * 1024
    min_duration_seconds: float = 1.0
    max_duration_seconds: float = 30.0
    min_rms: float = 1e-4
    min_peak: float = 1e-3
    min_active_seconds: float = 0.45


@dataclass(frozen=True)
class DecisionPolicy:
    """Three-way abstention policy operating on calibrated P(fake)."""

    policy_id: str = "legacy-symmetric-abstention-v1"
    real_max_p_fake: float = 0.35
    fake_min_p_fake: float = 0.65
    provenance: str = (
        "Legacy symmetric abstention band; not selected on the frozen test set. "
        "Replace only from validation data under the v2 protocol."
    )

    def __post_init__(self) -> None:
        if not 0.0 <= self.real_max

In [26]:
from pathlib import Path

path = Path("voice_guard_core.py")
source = path.read_text()

old = '''    min_active_seconds: float = 0.45


@dataclass(frozen=True)
class DecisionPolicy:
'''

new = '''    min_active_seconds: float = 0.45


@dataclass(frozen=True)
class CallInputPolicy:
    """Input limits for long phone-call recordings."""

    max_bytes: int = 200 * 1024 * 1024
    min_duration_seconds: float = 1.0
    max_duration_seconds: float = 900.0


@dataclass(frozen=True)
class DecisionPolicy:
'''

if source.count(old) != 1:
    raise RuntimeError("Expected insertion point not found exactly once. No changes made.")

path.write_text(source.replace(old, new, 1))
print("CallInputPolicy added successfully.")

CallInputPolicy added successfully.


In [27]:
from voice_guard_core import AudioInputPolicy, CallInputPolicy, DecisionPolicy

print("Single-clip maximum:", AudioInputPolicy().max_duration_seconds, "seconds")
print("Single-clip upload limit:", AudioInputPolicy().max_bytes // (1024 * 1024), "MB")
print("Call maximum:", CallInputPolicy().max_duration_seconds, "seconds")
print("Call upload limit:", CallInputPolicy().max_bytes // (1024 * 1024), "MB")
print("Decision thresholds:", DecisionPolicy().real_max_p_fake, DecisionPolicy().fake_min_p_fake)

Single-clip maximum: 30.0 seconds
Single-clip upload limit: 25 MB
Call maximum: 900.0 seconds
Call upload limit: 200 MB
Decision thresholds: 0.35 0.65


In [28]:
!pytest -q

.......................                                                  [100%]
23 passed in 2.17s


In [29]:
from pathlib import Path

path = Path("call_analysis.py")

if path.exists():
    raise FileExistsError("call_analysis.py already exists. No changes made.")

source = '''"""Long-call speech detection and window segmentation for Voice Guard."""

from __future__ import annotations

from dataclasses import dataclass
from typing import List

import numpy as np


@dataclass(frozen=True)
class SpeechRegion:
    """Speech-active interval in the original recording timeline."""

    start_seconds: float
    end_seconds: float


@dataclass(frozen=True)
class AnalysisWindow:
    """A model-analysis window with its original recording timestamps."""

    start_seconds: float
    end_seconds: float
    start_sample: int
    end_sample: int


def detect_speech_regions(
    waveform: np.ndarray,
    sample_rate: int,
    frame_duration: float = 0.025,
    hop_duration: float = 0.010,
    merge_gap_seconds: float = 0.30,
    min_region_seconds: float = 0.15,
    energy_threshold: float | None = None,
) -> List[SpeechRegion]:
    """Find speech-active regions using a lightweight adaptive energy gate.

    This is an energy-based detector, not a trained voice activity detector.
    It preserves timestamps relative to the original recording.
    """
    samples = np.asarray(waveform, dtype=np.float32).reshape(-1)

    if sample_rate <= 0:
        raise ValueError("sample_rate must be positive.")
    if samples.size == 0:
        return []
    if not np.isfinite(samples).all():
        raise ValueError("waveform contains non-finite values.")
    if frame_duration <= 0 or hop_duration <= 0:
        raise ValueError("Frame and hop durations must be positive.")

    frame_size = max(1, int(round(sample_rate * frame_duration)))
    hop_size = max(1, int(round(sample_rate * hop_duration)))

    if samples.size < frame_size:
        starts = [0]
    else:
        starts = range(0, samples.size - frame_size + 1, hop_size)

    frame_rms = np.asarray(
        [
            np.sqrt(np.mean(
                np.square(samples[start : start + frame_size], dtype=np.float64)
            ))
            for start in starts
        ],
        dtype=np.float64,
    )

    if energy_threshold is None:
        noise_floor = float(np.percentile(frame_rms, 20))
        peak_frame = float(np.max(frame_rms))
        threshold = max(1e-4, min(noise_floor * 2.0, peak_frame * 0.10))
    else:
        if energy_threshold < 0:
            raise ValueError("energy_threshold cannot be negative.")
        threshold = float(energy_threshold)

    active = frame_rms >= threshold
    active_indices = np.flatnonzero(active)
    if active_indices.size == 0:
        return []

    merge_frames = int(round(merge_gap_seconds / hop_duration))
    groups = []
    group_start = previous = int(active_indices[0])

    for index in active_indices[1:]:
        index = int(index)
        if index - previous - 1 > merge_frames:
            groups.append((group_start, previous))
            group_start = index
        previous = index
    groups.append((group_start, previous))

    regions = []
    for first_frame, last_frame in groups:
        start_sample = first_frame * hop_size
        end_sample = min(samples.size, last_frame * hop_size + frame_size)
        start_seconds = start_sample / float(sample_rate)
        end_seconds = end_sample / float(sample_rate)

        if end_seconds - start_seconds >= min_region_seconds:
            regions.append(
                SpeechRegion(
                    start_seconds=start_seconds,
                    end_seconds=end_seconds,
                )
            )

    return regions


def build_analysis_windows(
    regions: List[SpeechRegion],
    sample_rate: int,
    window_seconds: float = 6.0,
    hop_seconds: float = 3.0,
    min_trailing_seconds: float = 2.0,
) -> List[AnalysisWindow]:
    """Create overlapping analysis windows within speech regions."""
    if sample_rate <= 0:
        raise ValueError("sample_rate must be positive.")
    if window_seconds <= 0 or hop_seconds <= 0 or min_trailing_seconds < 0:
        raise ValueError("Window, hop, and trailing durations must be valid.")

    windows = []
    for region in regions:
        if region.start_seconds < 0 or region.end_seconds < region.start_seconds:
            raise ValueError("Invalid speech region.")

        start = region.start_seconds
        while start < region.end_seconds:
            duration = min(window_seconds, region.end_seconds - start)
            if duration < min_trailing_seconds:
                break

            start_sample = int(round(start * sample_rate))
            end_sample = int(round((start + duration) * sample_rate))

            windows.append(
                AnalysisWindow(
                    start_seconds=start,
                    end_seconds=start + duration,
                    start_sample=start_sample,
                    end_sample=end_sample,
                )
            )
            start += hop_seconds

    return windows
'''

path.write_text(source)
print("Created call_analysis.py with speech-region detection and window segmentation.")

Created call_analysis.py with speech-region detection and window segmentation.


In [30]:
from pathlib import Path

path = Path("tests/test_call_analysis.py")

if path.exists():
    raise FileExistsError("tests/test_call_analysis.py already exists. No changes made.")

source = '''import numpy as np

from call_analysis import (
    SpeechRegion,
    build_analysis_windows,
    detect_speech_regions,
)


def make_tone(duration, sample_rate=16000, amplitude=0.08):
    time = np.arange(int(duration * sample_rate), dtype=np.float32) / sample_rate
    return amplitude * np.sin(2 * np.pi * 220 * time)


def test_silence_produces_no_speech_regions():
    silence = np.zeros(5 * 16000, dtype=np.float32)

    regions = detect_speech_regions(silence, 16000)

    assert regions == []


def test_quiet_speech_is_detected():
    quiet_speech = make_tone(2.0, amplitude=0.003)

    regions = detect_speech_regions(quiet_speech, 16000)

    assert len(regions) == 1
    assert regions[0].end_seconds - regions[0].start_seconds >= 1.8


def test_silence_gap_creates_separate_speech_regions():
    first = make_tone(2.0)
    silence = np.zeros(int(0.8 * 16000), dtype=np.float32)
    second = make_tone(2.0)
    waveform = np.concatenate([first, silence, second])

    regions = detect_speech_regions(waveform, 16000)

    assert len(regions) == 2
    assert regions[0].end_seconds < regions[1].start_seconds
    assert 0.5 <= regions[1].start_seconds - regions[0].end_seconds <= 1.1


def test_windows_use_six_seconds_and_three_second_hop():
    regions = [SpeechRegion(0.0, 13.0)]

    windows = build_analysis_windows(regions, 16000)

    assert [window.start_seconds for window in windows] == [0.0, 3.0, 6.0, 9.0]
    assert windows[0].end_seconds == 6.0
    assert windows[1].start_seconds == 3.0
    assert windows[1].end_seconds == 9.0


def test_trailing_window_shorter_than_two_seconds_is_dropped():
    regions = [SpeechRegion(0.0, 7.0)]

    windows = build_analysis_windows(regions, 16000)

    assert all(
        window.end_seconds - window.start_seconds >= 2.0
        for window in windows
    )
    assert windows[-1].end_seconds == 6.0
'''

path.write_text(source)
print("Created tests/test_call_analysis.py.")

Created tests/test_call_analysis.py.


In [32]:
import os
from pathlib import Path

print("Current directory:", os.getcwd())
print("File exists:", Path("call_analysis.py").exists())
print("Absolute path:", Path("call_analysis.py").resolve())
print("Python import path:", os.getcwd() in __import__("sys").path)
print("Matching files:", list(Path(".").glob("call_analysis*")))

Current directory: /content/indic-fake-voice-detection
File exists: True
Absolute path: /content/indic-fake-voice-detection/call_analysis.py
Python import path: False
Matching files: [PosixPath('call_analysis.py')]


In [35]:
from pathlib import Path

test_file = Path("tests/test_call_analysis.py")
content = test_file.read_text()

content = content.replace(
    "    assert windows[-1].end_seconds == 6.0",
    "    assert len(windows) == 2\n"
    "    assert windows[-1].start_seconds == 3.0\n"
    "    assert windows[-1].end_seconds == 7.0"
)

test_file.write_text(content)
print("Corrected the trailing-window test expectation.")

Corrected the trailing-window test expectation.


In [36]:
import sys
!{sys.executable} -m pytest -q tests/test_call_analysis.py

.....                                                                    [100%]
5 passed in 0.18s


In [37]:
import sys
!{sys.executable} -m pytest -q

............................                                             [100%]
28 passed in 2.22s


In [38]:
!git status --short

 M voice_guard_core.py
?? call_analysis.py
?? tests/test_call_analysis.py


In [39]:
from pathlib import Path

path = Path("model_pipeline.py")
content = path.read_text()

for function_name in [
    "def decode_audio_bytes(",
    "def ensure_target_sample_rate(",
    "def load_inference_bundle(",
    "def score_waveform(",
]:
    position = content.find(function_name)
    print(f"\n{'=' * 70}\n{function_name}\n{'=' * 70}")
    print(content[position:position + 1800] if position >= 0 else "Not found")


def decode_audio_bytes(
def decode_audio_bytes(audio_bytes: bytes) -> Tuple[np.ndarray, int]:
    """
    Decode arbitrary audio bytes (wav/flac/mp3/ogg/m4a) to a mono float32 array.

    Tries soundfile, then torchaudio (with ffmpeg backend), then pydub.
    This is also used by decode_audio() for encoded dataset examples.
    """
    import io

    last_error: Optional[BaseException] = None

    try:
        import soundfile as sf

        data, sr = sf.read(io.BytesIO(audio_bytes), dtype="float32", always_2d=False)
        if getattr(data, "ndim", 1) > 1:
            data = data.mean(axis=1)          # downmix to mono
        return np.asarray(data, dtype=np.float32), int(sr)
    except Exception as exc:  # noqa: BLE001
        last_error = exc

    try:
        import torchaudio

        waveform, sr = torchaudio.load(io.BytesIO(audio_bytes), format=None)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0)
        return waveform.numpy().astype(np.float32

In [40]:
from pathlib import Path

content = Path("model_pipeline.py").read_text()
start = content.find("@dataclass\nclass InferenceBundle")
end = content.find("\ndef load_inference_bundle(", start)

print(content[start:end])

@dataclass
class InferenceBundle:
    """Everything needed to score a clip, loaded once and cached."""

    head: nn.Module
    temperature: float
    checkpoint_path: str
    checkpoint_epoch: int
    config: Dict[str, Any]
    summary_path: Optional[str]
    calibration_ece_before: Optional[float]
    calibration_ece_after: Optional[float]
    validation_eer_threshold_bonafide: Optional[float]


# Keyed cache so tests and services using more than one checkpoint directory do
# not silently receive whichever model happened to load first.
_INFERENCE_BUNDLES: Dict[Tuple[str, str], InferenceBundle] = {}




In [41]:
from pathlib import Path

content = Path("call_analysis.py").read_text()
print(content)

"""Long-call speech detection and window segmentation for Voice Guard."""

from __future__ import annotations

from dataclasses import dataclass
from typing import List

import numpy as np


@dataclass(frozen=True)
class SpeechRegion:
    """Speech-active interval in the original recording timeline."""

    start_seconds: float
    end_seconds: float


@dataclass(frozen=True)
class AnalysisWindow:
    """A model-analysis window with its original recording timestamps."""

    start_seconds: float
    end_seconds: float
    start_sample: int
    end_sample: int


def detect_speech_regions(
    waveform: np.ndarray,
    sample_rate: int,
    frame_duration: float = 0.025,
    hop_duration: float = 0.010,
    merge_gap_seconds: float = 0.30,
    min_region_seconds: float = 0.15,
    energy_threshold: float | None = None,
) -> List[SpeechRegion]:
    """Find speech-active regions using a lightweight adaptive energy gate.

    This is an energy-based detector, not a trained voice activity de

In [42]:
from pathlib import Path

path = Path("call_analysis.py")
with path.open("a", encoding="utf-8") as file:
    file.write("\n\n" + '''@dataclass(frozen=True)
class WindowScore:
    """Fake-voice score for one analysis window."""

    start_seconds: float
    end_seconds: float
    raw_logit: float
    p_fake: float
    verdict: str


@dataclass(frozen=True)
class CallAnalysisReport:
    """Complete analysis results for one phone-call recording."""

    duration_seconds: float
    speech_duration_seconds: float
    windows_analysed: int
    flagged_windows: int
    flagged_share: float
    highest_p_fake: float
    highest_score_timestamp: float
    window_scores: List[WindowScore]
    flagged_ranges: List[SpeechRegion]
''')

print("Added WindowScore and CallAnalysisReport.")

Added WindowScore and CallAnalysisReport.


In [43]:
from pathlib import Path

content = Path("voice_guard_core.py").read_text()

for term in ["def verdict_from_p_fake", "class Verdict"]:
    position = content.find(term)
    if position >= 0:
        print(content[position:position + 1600])

In [44]:
from pathlib import Path

for filename in ["model_pipeline.py", "voice_guard_core.py"]:
    content = Path(filename).read_text()
    print(f"\n--- Searching {filename} ---")

    for line_number, line in enumerate(content.splitlines(), start=1):
        if "verdict_from_p_fake" in line or "real_max_p_fake" in line or "fake_min_p_fake" in line:
            print(f"{line_number}: {line.strip()}")


--- Searching model_pipeline.py ---
2855: VERDICT_REAL_MAX_P_FAKE = DEFAULT_DECISION_POLICY.real_max_p_fake
2856: VERDICT_FAKE_MIN_P_FAKE = DEFAULT_DECISION_POLICY.fake_min_p_fake
2859: def verdict_from_p_fake(
2865: real_max == DEFAULT_DECISION_POLICY.real_max_p_fake
2866: and fake_min == DEFAULT_DECISION_POLICY.fake_min_p_fake
2875: real_max_p_fake=real_max,
2876: fake_min_p_fake=fake_min,
3090: verdict = verdict_from_p_fake(p_fake)
3112: "real_max_p_fake": VERDICT_REAL_MAX_P_FAKE,
3113: "fake_min_p_fake": VERDICT_FAKE_MIN_P_FAKE,
3529: "verdict": verdict_from_p_fake(float(p_fake[i])),

--- Searching voice_guard_core.py ---
67: real_max_p_fake: float = 0.35
68: fake_min_p_fake: float = 0.65
75: if not 0.0 <= self.real_max_p_fake < self.fake_min_p_fake <= 1.0:
186: if p_fake < policy.real_max_p_fake:
188: if p_fake > policy.fake_min_p_fake:


In [45]:
from pathlib import Path

content = Path("model_pipeline.py").read_text()
start = content.find("def verdict_from_p_fake(")

print(content[start:start + 1300])

def verdict_from_p_fake(
    p_fake: float,
    real_max: float = VERDICT_REAL_MAX_P_FAKE,
    fake_min: float = VERDICT_FAKE_MIN_P_FAKE,
) -> str:
    if (
        real_max == DEFAULT_DECISION_POLICY.real_max_p_fake
        and fake_min == DEFAULT_DECISION_POLICY.fake_min_p_fake
    ):
        return verdict_from_probability(p_fake)
    from voice_guard_core import DecisionPolicy

    return verdict_from_probability(
        p_fake,
        DecisionPolicy(
            policy_id="caller-supplied",
            real_max_p_fake=real_max,
            fake_min_p_fake=fake_min,
            provenance="Caller-supplied thresholds.",
        ),
    )


def decode_audio_bytes(audio_bytes: bytes) -> Tuple[np.ndarray, int]:
    """
    Decode arbitrary audio bytes (wav/flac/mp3/ogg/m4a) to a mono float32 array.

    Tries soundfile, then torchaudio (with ffmpeg backend), then pydub.
    This is also used by decode_audio() for encoded dataset examples.
    """
    import io

    last_error: Optiona

In [46]:
from pathlib import Path

content = Path("voice_guard_core.py").read_text()
start = content.find("def verdict_from_probability(")

print(content[start:start + 1100])

def verdict_from_probability(
    p_fake: float,
    policy: DecisionPolicy = DEFAULT_DECISION_POLICY,
) -> str:
    if not math.isfinite(p_fake) or not 0.0 <= p_fake <= 1.0:
        raise ValueError("p_fake must be a finite probability in [0, 1].")
    if p_fake < policy.real_max_p_fake:
        return "real"
    if p_fake > policy.fake_min_p_fake:
        return "fake"
    return "uncertain"


def normalize_language_hint(language_hint: str | None) -> str:
    """Accept only a declared dataset language; never pretend it was detected."""
    if language_hint is None or not str(language_hint).strip():
        return "unknown"
    value = str(language_hint).strip()
    by_lower = {language.lower(): language for language in SUPPORTED_LANGUAGES}
    try:
        return by_lower[value.lower()]
    except KeyError as exc:
        raise ValueError(
            f"Unsupported language hint {value!r}. Choose one of the 16 supported languages."
        ) from exc



In [47]:
from pathlib import Path

path = Path("call_analysis.py")

with path.open("a", encoding="utf-8") as file:
    file.write("\n\n" + '''def merge_flagged_ranges(
    window_scores: List[WindowScore],
) -> List[SpeechRegion]:
    """Merge overlapping or adjacent windows classified as fake."""
    flagged = sorted(
        (score for score in window_scores if score.verdict == "fake"),
        key=lambda score: score.start_seconds,
    )

    if not flagged:
        return []

    merged = []
    current_start = flagged[0].start_seconds
    current_end = flagged[0].end_seconds

    for score in flagged[1:]:
        if score.start_seconds <= current_end:
            current_end = max(current_end, score.end_seconds)
        else:
            merged.append(
                SpeechRegion(current_start, current_end)
            )
            current_start = score.start_seconds
            current_end = score.end_seconds

    merged.append(SpeechRegion(current_start, current_end))
    return merged
''')

print("Added merge_flagged_ranges().")

Added merge_flagged_ranges().


In [48]:
from call_analysis import WindowScore, merge_flagged_ranges

scores = [
    WindowScore(0, 6, 1.0, 0.80, "fake"),
    WindowScore(3, 9, 1.2, 0.85, "fake"),
    WindowScore(9, 15, 0.1, 0.50, "uncertain"),
    WindowScore(15, 21, -1.0, 0.20, "real"),
    WindowScore(24, 30, 1.1, 0.75, "fake"),
]

ranges = merge_flagged_ranges(scores)

print("Merged ranges:")
for region in ranges:
    print(f"{region.start_seconds:.1f}s - {region.end_seconds:.1f}s")

assert len(ranges) == 2
assert (ranges[0].start_seconds, ranges[0].end_seconds) == (0, 9)
assert (ranges[1].start_seconds, ranges[1].end_seconds) == (24, 30)

print("All merge checks passed.")

Merged ranges:
0.0s - 9.0s
24.0s - 30.0s
All merge checks passed.


In [49]:
import model_pipeline

for name in [
    "decode_audio_bytes",
    "ensure_target_sample_rate",
    "load_inference_bundle",
    "score_waveform",
    "verdict_from_p_fake",
    "TARGET_SAMPLE_RATE",
]:
    print(f"{name}: {hasattr(model_pipeline, name)}")

decode_audio_bytes: True
ensure_target_sample_rate: True
load_inference_bundle: True
score_waveform: True
verdict_from_p_fake: True
TARGET_SAMPLE_RATE: True


In [50]:
from pathlib import Path

path = Path("call_analysis.py")

with path.open("a", encoding="utf-8") as file:
    file.write("\n\n" + '''def analyze_call(
    audio_bytes: bytes,
    checkpoint_dir: str | None = None,
    window_seconds: float = 6.0,
    hop_seconds: float = 3.0,
    progress_callback=None,
) -> CallAnalysisReport:
    """Analyse a long phone call using the existing Voice Guard model."""
    import model_pipeline

    if not audio_bytes:
        raise ValueError("No audio data was supplied.")

    waveform, original_sample_rate = model_pipeline.decode_audio_bytes(
        audio_bytes
    )

    waveform = np.asarray(waveform, dtype=np.float32).reshape(-1)

    if waveform.size == 0:
        raise ValueError("The decoded recording is empty.")
    if original_sample_rate <= 0:
        raise ValueError("Invalid audio sample rate.")
    if not np.isfinite(waveform).all():
        raise ValueError("The recording contains non-finite audio values.")

    duration_seconds = waveform.size / float(original_sample_rate)

    if duration_seconds > 900.0:
        raise ValueError("Call recordings cannot exceed 15 minutes.")
    if duration_seconds < 1.0:
        raise ValueError("The recording must be at least 1 second long.")

    waveform = model_pipeline.ensure_target_sample_rate(
        waveform,
        original_sample_rate,
        model_pipeline.TARGET_SAMPLE_RATE,
    )
    sample_rate = model_pipeline.TARGET_SAMPLE_RATE

    regions = detect_speech_regions(waveform, sample_rate)
    windows = build_analysis_windows(
        regions,
        sample_rate,
        window_seconds=window_seconds,
        hop_seconds=hop_seconds,
    )

    if not windows:
        raise ValueError(
            "No speech-active regions long enough for analysis were detected."
        )

    if checkpoint_dir is None:
        bundle = model_pipeline.load_inference_bundle()
    else:
        bundle = model_pipeline.load_inference_bundle(
            checkpoint_dir=checkpoint_dir
        )

    window_scores = []

    for index, window in enumerate(windows):
        segment = waveform[window.start_sample:window.end_sample]

        raw_logit, p_fake = model_pipeline.score_waveform(
            segment,
            sample_rate,
            bundle=bundle,
        )
        verdict = model_pipeline.verdict_from_p_fake(p_fake)

        window_scores.append(
            WindowScore(
                start_seconds=window.start_seconds,
                end_seconds=window.end_seconds,
                raw_logit=raw_logit,
                p_fake=p_fake,
                verdict=verdict,
            )
        )

        if progress_callback is not None:
            progress_callback((index + 1) / len(windows))

    flagged_windows = sum(
        score.verdict == "fake" for score in window_scores
    )

    flagged_share = flagged_windows / len(window_scores)

    highest_score = max(window_scores, key=lambda score: score.p_fake)

    speech_duration_seconds = sum(
        region.end_seconds - region.start_seconds
        for region in regions
    )

    return CallAnalysisReport(
        duration_seconds=duration_seconds,
        speech_duration_seconds=speech_duration_seconds,
        windows_analysed=len(window_scores),
        flagged_windows=flagged_windows,
        flagged_share=flagged_share,
        highest_p_fake=highest_score.p_fake,
        highest_score_timestamp=highest_score.start_seconds,
        window_scores=window_scores,
        flagged_ranges=merge_flagged_ranges(window_scores),
    )
''')

print("Added analyze_call().")

Added analyze_call().


In [51]:
import inspect
import voice_guard_core

print(inspect.signature(voice_guard_core.validate_encoded_audio_size))
print(inspect.getsource(voice_guard_core.validate_encoded_audio_size))

(audio_bytes: 'bytes', policy: 'AudioInputPolicy' = AudioInputPolicy(max_bytes=26214400, min_duration_seconds=1.0, max_duration_seconds=30.0, min_rms=0.0001, min_peak=0.001, min_active_seconds=0.45)) -> 'None'
def validate_encoded_audio_size(
    audio_bytes: bytes,
    policy: AudioInputPolicy = DEFAULT_AUDIO_POLICY,
) -> None:
    if not isinstance(audio_bytes, (bytes, bytearray)):
        raise AudioValidationError("The uploaded audio could not be read.")
    if not audio_bytes:
        raise AudioValidationError("The uploaded audio file is empty.")
    if len(audio_bytes) > policy.max_bytes:
        limit_mb = policy.max_bytes / (1024 * 1024)
        raise AudioValidationError(
            f"The audio file is too large. Upload a file no larger than {limit_mb:.0f} MB."
        )



In [52]:
from pathlib import Path

path = Path("call_analysis.py")
content = path.read_text()

old = '''    if not audio_bytes:
        raise ValueError("No audio data was supplied.")

    waveform, original_sample_rate = model_pipeline.decode_audio_bytes(
'''

new = '''    from voice_guard_core import (
        CallInputPolicy,
        validate_encoded_audio_size,
    )

    validate_encoded_audio_size(
        audio_bytes,
        policy=CallInputPolicy(),
    )

    waveform, original_sample_rate = model_pipeline.decode_audio_bytes(
'''

if old not in content:
    raise RuntimeError("Expected analyze_call section was not found. No changes made.")

path.write_text(content.replace(old, new, 1))
print("Added the 200 MB call-upload validation before decoding.")

Added the 200 MB call-upload validation before decoding.


In [53]:
import sys
!{sys.executable} -m py_compile call_analysis.py voice_guard_core.py
!{sys.executable} -m pytest -q

............................                                             [100%]
28 passed in 2.25s


In [54]:
from pathlib import Path

print(Path("tests/test_call_analysis.py").read_text())

import numpy as np

from call_analysis import (
    SpeechRegion,
    build_analysis_windows,
    detect_speech_regions,
)


def make_tone(duration, sample_rate=16000, amplitude=0.08):
    time = np.arange(int(duration * sample_rate), dtype=np.float32) / sample_rate
    return amplitude * np.sin(2 * np.pi * 220 * time)


def test_silence_produces_no_speech_regions():
    silence = np.zeros(5 * 16000, dtype=np.float32)

    regions = detect_speech_regions(silence, 16000)

    assert regions == []


def test_quiet_speech_is_detected():
    quiet_speech = make_tone(2.0, amplitude=0.003)

    regions = detect_speech_regions(quiet_speech, 16000)

    assert len(regions) == 1
    assert regions[0].end_seconds - regions[0].start_seconds >= 1.8


def test_silence_gap_creates_separate_speech_regions():
    first = make_tone(2.0)
    silence = np.zeros(int(0.8 * 16000), dtype=np.float32)
    second = make_tone(2.0)
    waveform = np.concatenate([first, silence, second])

    regions = detect_spe

In [55]:
from pathlib import Path

path = Path("tests/test_call_analysis.py")

with path.open("a", encoding="utf-8") as file:
    file.write("\n\n" + '''import pytest

from voice_guard_core import AudioInputPolicy, CallInputPolicy


def test_call_and_single_clip_upload_policies_remain_separate():
    assert CallInputPolicy().max_bytes == 200 * 1024 * 1024
    assert CallInputPolicy().max_duration_seconds == 900.0

    assert AudioInputPolicy().max_bytes == 25 * 1024 * 1024
    assert AudioInputPolicy().max_duration_seconds == 30.0


def test_recording_over_fifteen_minutes_is_rejected(monkeypatch):
    import model_pipeline
    from call_analysis import analyze_call

    monkeypatch.setattr(
        model_pipeline,
        "decode_audio_bytes",
        lambda audio_bytes: (
            np.zeros(901, dtype=np.float32),
            1,
        ),
    )

    def unexpected_resampling(*args, **kwargs):
        raise AssertionError("Resampling should not run for an overlong call.")

    monkeypatch.setattr(
        model_pipeline,
        "ensure_target_sample_rate",
        unexpected_resampling,
    )

    with pytest.raises(ValueError, match="15 minutes"):
        analyze_call(b"test-audio")
''')

print("Added two call-policy and duration-validation tests.")

Added two call-policy and duration-validation tests.


In [56]:
import sys
!{sys.executable} -m pytest -q tests/test_call_analysis.py

.......                                                                  [100%]
7 passed in 2.01s


In [58]:
import importlib
import call_analysis

importlib.reload(call_analysis)

print("analyze_call exists:", hasattr(call_analysis, "analyze_call"))
print("CallAnalysisReport exists:", hasattr(call_analysis, "CallAnalysisReport"))

if hasattr(call_analysis, "analyze_call"):
    import inspect
    print("Signature:", inspect.signature(call_analysis.analyze_call))

analyze_call exists: True
CallAnalysisReport exists: True
Signature: (audio_bytes: 'bytes', checkpoint_dir: 'str | None' = None, window_seconds: 'float' = 6.0, hop_seconds: 'float' = 3.0, progress_callback=None) -> 'CallAnalysisReport'


In [59]:
import inspect
from call_analysis import analyze_call, CallAnalysisReport

print("analyze_call:", inspect.signature(analyze_call))
print("Report fields:", list(CallAnalysisReport.__dataclass_fields__.keys()))

analyze_call: (audio_bytes: 'bytes', checkpoint_dir: 'str | None' = None, window_seconds: 'float' = 6.0, hop_seconds: 'float' = 3.0, progress_callback=None) -> 'CallAnalysisReport'
Report fields: ['duration_seconds', 'speech_duration_seconds', 'windows_analysed', 'flagged_windows', 'flagged_share', 'highest_p_fake', 'highest_score_timestamp', 'window_scores', 'flagged_ranges']


In [60]:
import numpy as np
import model_pipeline
from call_analysis import analyze_call

# Simulate a 15-second recording with a short silence gap
sample_rate = 16000
waveform = np.concatenate([
    np.ones(8 * sample_rate, dtype=np.float32) * 0.1,
    np.zeros(1 * sample_rate, dtype=np.float32),
    np.ones(6 * sample_rate, dtype=np.float32) * 0.1,
])

# Mock the existing pipeline functions
original_decode = model_pipeline.decode_audio_bytes
original_resample = model_pipeline.ensure_target_sample_rate
original_load = model_pipeline.load_inference_bundle
original_score = model_pipeline.score_waveform
original_verdict = model_pipeline.verdict_from_p_fake

try:
    model_pipeline.decode_audio_bytes = lambda audio_bytes: (waveform, sample_rate)
    model_pipeline.ensure_target_sample_rate = lambda audio, sr, target: audio
    model_pipeline.load_inference_bundle = lambda **kwargs: "mock_bundle"

    model_pipeline.score_waveform = (
        lambda segment, sr, bundle=None: (1.0, 0.8)
    )
    model_pipeline.verdict_from_p_fake = lambda p: "fake" if p >= 0.65 else "real"

    progress = []

    report = analyze_call(
        b"mock-audio",
        progress_callback=progress.append,
    )

    print("Duration:", report.duration_seconds)
    print("Speech duration:", report.speech_duration_seconds)
    print("Windows analysed:", report.windows_analysed)
    print("Flagged windows:", report.flagged_windows)
    print("Flagged share:", report.flagged_share)
    print("Highest fake score:", report.highest_p_fake)
    print("Flagged ranges:", report.flagged_ranges)
    print("Final progress:", progress[-1] if progress else "No progress")

finally:
    model_pipeline.decode_audio_bytes = original_decode
    model_pipeline.ensure_target_sample_rate = original_resample
    model_pipeline.load_inference_bundle = original_load
    model_pipeline.score_waveform = original_score
    model_pipeline.verdict_from_p_fake = original_verdict

Duration: 15.0
Speech duration: 14.03
Windows analysed: 5
Flagged windows: 5
Flagged share: 1.0
Highest fake score: 0.8
Flagged ranges: [SpeechRegion(start_seconds=0.0, end_seconds=8.015), SpeechRegion(start_seconds=8.98, end_seconds=14.995)]
Final progress: 1.0


In [61]:
from pathlib import Path

files = list(Path(".").glob("*.mp4"))

if files:
    for file in files:
        print("Found:", file.resolve())
        print("Size:", round(file.stat().st_size / 1024, 2), "KB")
else:
    print("No MP4 file found in the current directory.")

No MP4 file found in the current directory.


In [62]:
from pathlib import Path

matches = list(Path("/content").rglob("WhatsApp Audio*.mp4"))

if matches:
    for file in matches:
        print("Found:", file)
        print("Size:", round(file.stat().st_size / 1024, 2), "KB")
else:
    print("File not found under /content. Please check the Colab Files panel.")

Found: /content/WhatsApp Audio 2026-11-16 at 6.35.58 PM.mp4
Size: 806.47 KB


In [63]:
from pathlib import Path
import model_pipeline
import numpy as np

audio_path = Path(
    "/content/WhatsApp Audio 2026-11-16 at 6.35.58 PM.mp4"
)

audio_bytes = audio_path.read_bytes()

try:
    waveform, sample_rate = model_pipeline.decode_audio_bytes(audio_bytes)

    waveform = np.asarray(waveform)

    print("DECODING SUCCESSFUL")
    print("Sample rate:", sample_rate)
    print("Duration:", round(waveform.size / sample_rate, 2), "seconds")
    print("Waveform shape:", waveform.shape)
    print("Finite values:", np.isfinite(waveform).all())

except Exception as error:
    print("DECODING FAILED")
    print(type(error).__name__ + ":", error)

DECODING SUCCESSFUL
Sample rate: 48000
Duration: 50.54 seconds
Waveform shape: (1, 2425856)
Finite values: True


In [64]:
from pathlib import Path
from call_analysis import analyze_call

audio_path = Path(
    "/content/WhatsApp Audio 2026-11-16 at 6.35.58 PM.mp4"
)

audio_bytes = audio_path.read_bytes()

progress_values = []

report = analyze_call(
    audio_bytes,
    progress_callback=progress_values.append,
)

print("\nREAL MODEL ANALYSIS")
print("-------------------")
print("Duration:", round(report.duration_seconds, 2), "seconds")
print("Speech duration:", round(report.speech_duration_seconds, 2), "seconds")
print("Windows analysed:", report.windows_analysed)
print("Flagged windows:", report.flagged_windows)
print("Flagged share:", round(report.flagged_share * 100, 2), "%")
print("Highest fake score:", round(report.highest_p_fake, 4))
print("Highest score timestamp:", round(report.highest_score_timestamp, 2), "seconds")
print("Flagged ranges:", report.flagged_ranges)
print("Progress:", progress_values[-1] if progress_values else "Unavailable")

Inference bundle loaded: /content/indic-fake-voice-detection/checkpoints/best_model.pt (epoch 11), temperature=1.2600


preprocessor_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.27GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



REAL MODEL ANALYSIS
-------------------
Duration: 50.54 seconds
Speech duration: 50.36 seconds
Windows analysed: 17
Flagged windows: 17
Flagged share: 100.0 %
Highest fake score: 0.9999
Highest score timestamp: 0.17 seconds
Flagged ranges: [SpeechRegion(start_seconds=0.17, end_seconds=50.535)]
Progress: 1.0


In [65]:
print("INDIVIDUAL WINDOW SCORES")
print("-" * 65)

for index, score in enumerate(report.window_scores, start=1):
    print(
        f"Window {index:02d} | "
        f"{score.start_seconds:6.2f}s - "
        f"{score.end_seconds:6.2f}s | "
        f"P(fake): {score.p_fake:.4f} | "
        f"Verdict: {score.verdict}"
    )

INDIVIDUAL WINDOW SCORES
-----------------------------------------------------------------
Window 01 |   0.17s -   6.17s | P(fake): 0.9999 | Verdict: fake
Window 02 |   3.17s -   9.17s | P(fake): 0.9999 | Verdict: fake
Window 03 |   6.17s -  12.17s | P(fake): 0.9999 | Verdict: fake
Window 04 |   9.17s -  15.17s | P(fake): 0.9999 | Verdict: fake
Window 05 |  12.17s -  18.17s | P(fake): 0.9999 | Verdict: fake
Window 06 |  15.17s -  21.17s | P(fake): 0.9991 | Verdict: fake
Window 07 |  18.17s -  24.17s | P(fake): 0.9979 | Verdict: fake
Window 08 |  21.17s -  27.17s | P(fake): 0.9997 | Verdict: fake
Window 09 |  24.17s -  30.17s | P(fake): 0.9998 | Verdict: fake
Window 10 |  27.17s -  33.17s | P(fake): 0.9998 | Verdict: fake
Window 11 |  30.17s -  36.17s | P(fake): 0.9996 | Verdict: fake
Window 12 |  33.17s -  39.17s | P(fake): 0.9994 | Verdict: fake
Window 13 |  36.17s -  42.17s | P(fake): 0.9995 | Verdict: fake
Window 14 |  39.17s -  45.17s | P(fake): 0.9983 | Verdict: fake
Window 15 |  

In [66]:
from pathlib import Path

matches = list(Path("/content").rglob("WhatsApp Audio 2026-11-16 at 6.43.34 PM.aac"))

if matches:
    for file in matches:
        print("Found:", file.resolve())
        print("Size:", round(file.stat().st_size / 1024, 2), "KB")
else:
    print("File not found. Please check the Colab Files panel.")

Found: /content/WhatsApp Audio 2026-11-16 at 6.43.34 PM.aac
Size: 844.66 KB


In [67]:
from pathlib import Path
import model_pipeline
import numpy as np

audio_path = Path(
    "/content/WhatsApp Audio 2026-11-16 at 6.43.34 PM.aac"
)

audio_bytes = audio_path.read_bytes()

try:
    waveform, sample_rate = model_pipeline.decode_audio_bytes(audio_bytes)
    waveform = np.asarray(waveform)

    print("DECODING SUCCESSFUL")
    print("Sample rate:", sample_rate)
    print("Duration:", round(waveform.shape[-1] / sample_rate, 2), "seconds")
    print("Waveform shape:", waveform.shape)
    print("Finite values:", np.isfinite(waveform).all())

except Exception as error:
    print("DECODING FAILED")
    print(type(error).__name__ + ":", error)

/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


DECODING SUCCESSFUL
Sample rate: 48000
Duration: 26.75 seconds
Waveform shape: (1284096,)
Finite values: True


In [68]:
from pathlib import Path
from call_analysis import analyze_call

audio_path = Path(
    "/content/WhatsApp Audio 2026-11-16 at 6.43.34 PM.aac"
)

progress_values = []

report_2 = analyze_call(
    audio_path.read_bytes(),
    progress_callback=progress_values.append,
)

print("\nREAL MODEL ANALYSIS")
print("-------------------")
print("Duration:", round(report_2.duration_seconds, 2), "seconds")
print("Speech duration:", round(report_2.speech_duration_seconds, 2), "seconds")
print("Windows analysed:", report_2.windows_analysed)
print("Flagged windows:", report_2.flagged_windows)
print("Flagged share:", round(report_2.flagged_share * 100, 2), "%")
print("Highest fake score:", round(report_2.highest_p_fake, 4))
print("Highest score timestamp:", round(report_2.highest_score_timestamp, 2), "seconds")
print("Flagged ranges:", report_2.flagged_ranges)
print("Progress:", progress_values[-1] if progress_values else "Unavailable")


REAL MODEL ANALYSIS
-------------------
Duration: 26.75 seconds
Speech duration: 26.38 seconds
Windows analysed: 9
Flagged windows: 9
Flagged share: 100.0 %
Highest fake score: 0.9999
Highest score timestamp: 24.37 seconds
Flagged ranges: [SpeechRegion(start_seconds=0.37, end_seconds=26.745)]
Progress: 1.0


In [69]:
from pathlib import Path
import model_pipeline
import inspect

audio_path = Path(
    "/content/WhatsApp Audio 2026-11-16 at 6.43.34 PM.aac"
)

audio_bytes = audio_path.read_bytes()

print("Original analysis function:")
print(inspect.signature(model_pipeline.analyze_bytes))

print("\nRunning original single-clip detector...")

result = model_pipeline.analyze_bytes(audio_bytes)

print("\nORIGINAL DETECTOR RESULT")
print("------------------------")
print("Result type:", type(result))
print("Result:", result)

Original analysis function:
(audio_bytes: 'bytes', language_hint: 'Optional[str]' = None, checkpoint_dir: 'str' = '/content/indic-fake-voice-detection/checkpoints') -> 'AnalysisResult'

Running original single-clip detector...

ORIGINAL DETECTOR RESULT
------------------------
Result type: <class 'model_pipeline.AnalysisResult'>
Result: AnalysisResult(verdict='fake', confidence=0.9993883974411212, detected_language='unknown', language_eer=None, raw_score=9.322735786437988, audio_hash='fb80a84f95144d832423c306ffd48ef8e5db17a63008b99ba7b6770f9552c274', metadata={'p_fake_calibrated': 0.9993883974411212, 'temperature': 1.260030746459961, 'thresholds': {'real_max_p_fake': 0.35, 'fake_min_p_fake': 0.65}, 'decision_policy_id': 'legacy-symmetric-abstention-v1', 'decision_policy_provenance': 'Legacy symmetric abstention band; not selected on the frozen test set. Replace only from validation data under the v2 protocol.', 'duration_seconds': 26.752, 'sample_rate_in': 48000, 'audio_quality': {'dur

In [70]:
from pathlib import Path
import json

project = Path("/content/indic-fake-voice-detection")

matches = list(project.rglob("step6_summary.json"))

print("Calibration reports found:", len(matches))

for path in matches:
    print("\nFILE:", path)
    with open(path, "r") as f:
        data = json.load(f)

    print(json.dumps(data, indent=2)[:12000])

Calibration reports found: 1

FILE: /content/indic-fake-voice-detection/checkpoints/step6_summary.json
{
  "config": {
    "hidden_dim": 256,
    "num_gat_layers": 2,
    "dropout": 0.3,
    "learning_rate": 0.0001,
    "weight_decay": 0.01,
    "num_epochs": 15,
    "batch_size": 16,
    "shuffle_train": true,
    "grad_clip": 0.0,
    "seed": 42
  },
  "num_train": 24877,
  "num_val": 3110,
  "loss_history": [
    0.3405411530521899,
    0.2955202936709909,
    0.19118477386318392,
    0.09860777470973467,
    0.06086201902004569,
    0.054444159730566835,
    0.05615249359983267,
    0.042109581662691103,
    0.03753932494846916,
    0.03969597780728179,
    0.04245650468488467,
    0.03222857704169767,
    0.02752233737404898,
    0.030865458250448292,
    0.031375557239494416
  ],
  "history": [
    {
      "epoch": 1,
      "train_loss": 0.3405411530521899,
      "val_loss": 0.9340847134590149,
      "val_eer": 0.22057877813504823
    },
    {
      "epoch": 2,
      "train_loss"

In [71]:
from pathlib import Path

project = Path("/content/indic-fake-voice-detection")

keywords = [
    "eval", "test", "calibr", "train",
    "dataset", "inference", "pipeline"
]

print("RELEVANT PROJECT FILES")
print("-" * 60)

for path in sorted(project.rglob("*")):
    if (
        path.is_file()
        and path.suffix in {".py", ".json", ".yaml", ".yml"}
        and any(word in path.name.lower() for word in keywords)
        and not any(part.startswith(".") for part in path.parts)
    ):
        print(path.relative_to(project))

RELEVANT PROJECT FILES
------------------------------------------------------------
evaluate_test_split.py
evaluation/clean/per_language_test_metrics.json
model_pipeline.py
test_analyze_bytes_e2e.py
tests/test_call_analysis.py
tests/test_codec_robustness.py
tests/test_evidence_report.py
tests/test_model_pipeline_audio.py
tests/test_split_protocol.py
tests/test_voice_guard_core.py


In [72]:
from pathlib import Path
import json

project = Path("/content/indic-fake-voice-detection")

# Inspect the evaluation script
script = project / "evaluate_test_split.py"

print("=" * 70)
print("EVALUATION SCRIPT")
print("=" * 70)
print(script.read_text()[:18000])

# Inspect the clean test metrics
metrics_file = (
    project / "evaluation" / "clean" / "per_language_test_metrics.json"
)

print("\n" + "=" * 70)
print("PER-LANGUAGE TEST METRICS")
print("=" * 70)

metrics = json.loads(metrics_file.read_text())
print(json.dumps(metrics, indent=2)[:12000])

EVALUATION SCRIPT
"""
Evaluate the trained model on the frozen TEST split (3,111 clips) — the split
Step 6 never touched during training or model selection.

Run this exactly once, after every Step 6 training/hyperparameter decision is
final. Its output is the first real, reportable test-set number for the
project. Re-running it after going back and changing something in Step 6 turns
the test set into a second validation set and invalidates that property.

Usage (Colab/Kaggle, after Step 6 has produced checkpoints/best_model.pt):

    !python evaluate_test_split.py \\
        --checkpoint-dir /content/drive/MyDrive/indic-fake-voice-detection/checkpoints \\
        --cache-dir /content/drive/MyDrive/indic-fake-voice-detection/feature_cache

or from a notebook cell:

    import evaluate_test_split as t
    result = t.main(
        checkpoint_dir="/content/drive/MyDrive/indic-fake-voice-detection/checkpoints",
        cache_dir="/content/drive/MyDrive/indic-fake-voice-detection/feature_ca

In [73]:
from pathlib import Path
import json

project = Path("/content/indic-fake-voice-detection")

# Read evaluation metrics
metrics_path = project / "evaluation/clean/per_language_test_metrics.json"
metrics = json.loads(metrics_path.read_text())

print("TEST METRICS")
print(json.dumps(metrics, indent=2)[:3500])

# Extract important evaluation-related lines from the script
script_path = project / "evaluate_test_split.py"
lines = script_path.read_text().splitlines()

keywords = [
    "threshold", "eer", "false", "genuine",
    "bonafide", "spoof", "speaker", "dataset",
    "calibration", "temperature", "split"
]

print("\nRELEVANT EVALUATION CODE")
print("-" * 50)

for number, line in enumerate(lines, start=1):
    if any(word in line.lower() for word in keywords):
        print(f"{number}: {line[:220]}")

TEST METRICS
[
  {
    "language": "Assamese",
    "test_clips": 200,
    "real_clips": 100,
    "fake_clips": 100,
    "eer": 0.0,
    "eer_percent": 0.0,
    "roc_auc": 1.0,
    "accuracy": 1.0,
    "accuracy_percent": 100.0,
    "precision": 1.0,
    "precision_percent": 100.0,
    "recall": 1.0,
    "recall_percent": 100.0,
    "f1": 1.0,
    "f1_percent": 100.0,
    "true_real_pred_real": 100,
    "true_real_pred_fake": 0,
    "true_fake_pred_real": 0,
    "true_fake_pred_fake": 100,
    "eer_threshold": 5.05246639251709
  },
  {
    "language": "Bengali",
    "test_clips": 180,
    "real_clips": 90,
    "fake_clips": 90,
    "eer": 0.0,
    "eer_percent": 0.0,
    "roc_auc": 1.0,
    "accuracy": 1.0,
    "accuracy_percent": 100.0,
    "precision": 1.0,
    "precision_percent": 100.0,
    "recall": 1.0,
    "recall_percent": 100.0,
    "f1": 1.0,
    "f1_percent": 100.0,
    "true_real_pred_real": 90,
    "true_real_pred_fake": 0,
    "true_fake_pred_real": 0,
    "true_fake_pred_

In [74]:
from pathlib import Path
import json

path = Path(
    "/content/indic-fake-voice-detection/evaluation/clean/per_language_test_metrics.json"
)

data = json.loads(path.read_text())

print("TOP-LEVEL STRUCTURE:", type(data).__name__)

if isinstance(data, dict):
    print("TOP-LEVEL KEYS:", list(data.keys()))

    for key, value in data.items():
        if isinstance(value, (int, float, str)):
            print(f"{key}: {value}")

print("\nLANGUAGE METRICS")
print("-" * 65)

for item in data if isinstance(data, list) else data.get("languages", []):
    print(
        f"{item.get('language', '?'):15} "
        f"n={item.get('test_clips', '?'):4} "
        f"EER={item.get('eer', '?')} "
        f"AUC={item.get('roc_auc', '?')} "
        f"Accuracy={item.get('accuracy', '?')} "
        f"Real→Fake={item.get('true_real_pred_fake', '?')} "
        f"Fake→Real={item.get('true_fake_pred_real', '?')}"
    )

TOP-LEVEL STRUCTURE: list

LANGUAGE METRICS
-----------------------------------------------------------------
Assamese        n= 200 EER=0.0 AUC=1.0 Accuracy=1.0 Real→Fake=0 Fake→Real=0
Bengali         n= 180 EER=0.0 AUC=1.0 Accuracy=1.0 Real→Fake=0 Fake→Real=0
Bodo            n= 200 EER=0.0 AUC=1.0 Accuracy=1.0 Real→Fake=0 Fake→Real=0
Dogri           n= 200 EER=0.0 AUC=1.0 Accuracy=1.0 Real→Fake=0 Fake→Real=0
English         n= 200 EER=0.0 AUC=1.0 Accuracy=0.875 Real→Fake=0 Fake→Real=25
Gujarati        n= 156 EER=0.0 AUC=1.0 Accuracy=0.8717948717948718 Real→Fake=0 Fake→Real=20
Hindi           n= 200 EER=0.0 AUC=1.0 Accuracy=0.925 Real→Fake=0 Fake→Real=15
Kannada         n= 200 EER=0.0 AUC=1.0 Accuracy=0.945 Real→Fake=0 Fake→Real=11
Malayalam       n= 201 EER=0.004950495049504955 AUC=0.9998019801980198 Accuracy=0.9502487562189055 Real→Fake=0 Fake→Real=10
Manipuri        n= 200 EER=0.010000000000000005 AUC=0.9986999999999999 Accuracy=0.965 Real→Fake=0 Fake→Real=7
Marathi         n= 201 

In [75]:
from pathlib import Path
import re

path = Path("/content/indic-fake-voice-detection/model_pipeline.py")
source = path.read_text().splitlines()

terms = [
    "def evaluate_dataset",
    "roc_auc_score",
    "roc_curve",
    "roc_auc",
    "eer_threshold",
    "true_real_pred_fake",
    "true_fake_pred_real",
    "decision_threshold",
]

for i, line in enumerate(source):
    if any(term in line for term in terms):
        start = max(0, i - 5)
        end = min(len(source), i + 15)

        print(f"\n--- Lines {start + 1}-{end} ---")
        for j in range(start, end):
            print(f"{j + 1}: {source[j]}")


--- Lines 2272-2291 ---
2272:     Equal Error Rate. Returns (eer, threshold).
2273: 
2274:     Sanity-checked: perfect classifier -> 0%, random -> ~50%, two unit-variance
2275:     Gaussians +/-1 apart -> ~15.87% theoretical, matched empirically.
2276:     """
2277:     from sklearn.metrics import roc_curve
2278: 
2279:     labels = np.concatenate([np.ones(len(bonafide_scores)), np.zeros(len(spoof_scores))])
2280:     scores = np.concatenate([bonafide_scores, spoof_scores])
2281: 
2282:     fpr, tpr, thresholds = roc_curve(labels, scores)
2283:     fnr = 1 - tpr
2284: 
2285:     eer_idx = np.nanargmin(np.abs(fpr - fnr))
2286:     eer = (fpr[eer_idx] + fnr[eer_idx]) / 2
2287:     return eer, thresholds[eer_idx]
2288: 
2289: 
2290: def eer_from_logits_and_labels(
2291:     logits: np.ndarray,

--- Lines 2277-2296 ---
2277:     from sklearn.metrics import roc_curve
2278: 
2279:     labels = np.concatenate([np.ones(len(bonafide_scores)), np.zeros(len(spoof_scores))])
2280:     scores = np

In [76]:
import inspect, model_pipeline
print(inspect.getsource(model_pipeline.evaluate_dataset)[:5000])

def evaluate_dataset(
    clip_ids: Sequence[str],
    labels: Sequence[int],
    checkpoint_dir: str = CHECKPOINT_DIR,
    cache_dir: str = FEATURE_CACHE_DIR,
    cache_split_name: str = "eval",
    language_lookup: Optional[Dict[str, str]] = None,
    decision_threshold: Optional[float] = None,
    batch_size: int = 16,
    extract_batch_size: int = 8,
    min_language_support: int = 1,
    write_language_eer_file: bool = False,
    include_per_clip: bool = False,
) -> dict:
    """
    Run the trained head over a set of clips and score it.

    Args:
        clip_ids, labels: parallel sequences; labels use the metadata convention
            (0 = real, 1 = fake). This is the ground truth used for every metric
            below — NOT the label baked into the feature cache, if one exists.
        cache_split_name: feature-cache bucket to build/reuse for this exact ID
            set. Using the same name across DIFFERENT id sets between calls is
            safe (load_features_for_ids 

In [77]:
import inspect, model_pipeline

lines = inspect.getsource(model_pipeline.evaluate_dataset).splitlines()
for i, line in enumerate(lines):
    if any(x in line.lower() for x in [
        "roc_curve", "eer", "confusion_matrix",
        "predicted", "decision_threshold", "p_fake"
    ]):
        print(f"{i+1}: {line.strip()}")

8: decision_threshold: Optional[float] = None,
12: write_language_eer_file: bool = False,
27: decision_threshold: cut on temperature-scaled P(fake) used ONLY for
28: accuracy/precision/recall/f1/confusion_matrix, which need a single
29: binary decision. If omitted, the validation EER operating threshold
34: (VERDICT_REAL_MAX_P_FAKE / VERDICT_FAKE_MIN_P_FAKE), and
35: * the EER operating threshold (returned explicitly in both
36: bonafide-score and fake-logit orientation) — EER itself does not
39: still reported (with their n) but their EER is set to None rather
41: write_language_eer_file: compatibility name. If True, writes a clearly
43: Per-language test EER is never consumed as a per-clip confidence.
45: evaluated clip (clip_id, language, true_label, predicted_label,
46: calibrated_p_fake, verdict) for error analysis. Off by default since
49: Returns a dict with pooled EER/accuracy/precision/recall/f1/confusion_matrix,
50: a per-language EER breakdown with support counts, and bookke

In [78]:
import model_pipeline

b = model_pipeline.load_inference_bundle()
print("Validation EER threshold (bonafide):", b.validation_eer_threshold_bonafide)
print("Temperature:", b.temperature)
print("Binary evaluation threshold:", model_pipeline.stable_sigmoid(
    -b.validation_eer_threshold_bonafide / b.temperature
))
print("Verdict thresholds:", 0.35, 0.65)

Validation EER threshold (bonafide): 2.1031923294067383
Temperature: 1.260030746459961
Binary evaluation threshold: 0.15853626739443824
Verdict thresholds: 0.35 0.65


In [79]:
import model_pipeline

b = model_pipeline.load_inference_bundle()
print("Checkpoint:", b.checkpoint_path if hasattr(b, "checkpoint_path") else "Not exposed")
print("Temperature:", b.temperature)
print("Validation threshold:", b.validation_eer_threshold_bonafide)

Checkpoint: /content/indic-fake-voice-detection/checkpoints/best_model.pt
Temperature: 1.260030746459961
Validation threshold: 2.1031923294067383


In [80]:
import inspect, model_pipeline

src = inspect.getsource(model_pipeline.score_waveform).splitlines()
for i, line in enumerate(src):
    print(f"{i+1}: {line.strip()}")

1: @torch.no_grad()
2: def score_waveform(
3: waveform: np.ndarray,
4: sample_rate: int = TARGET_SAMPLE_RATE,
5: bundle: Optional[InferenceBundle] = None,
6: ) -> Tuple[float, float]:
7: """
8: Score one waveform. Returns (raw_logit, calibrated_p_fake).
9: 
10: Higher logit = more fake, matching the training target (1 = fake).
11: """
12: bundle = bundle or load_inference_bundle()
13: 
14: feats = extract_features(waveform, sampling_rate=sample_rate)
15: if feats.shape[0] == 0:
16: raise ValueError("Frontend produced zero frames — clip is too short.")
17: 
18: x = feats.unsqueeze(0).float().to(DEVICE)
19: mask = torch.ones(1, feats.shape[0], device=DEVICE)
20: 
21: logit = bundle.head(x, mask=mask)
22: raw_logit = float(logit.reshape(-1)[0].item())
23: 
24: calibrated = raw_logit / bundle.temperature
25: p_fake = stable_sigmoid(calibrated)
26: 
27: return raw_logit, p_fake


In [81]:
import inspect, model_pipeline
print(inspect.getsource(model_pipeline.extract_features))

def extract_features(
    audio_array: np.ndarray,
    sampling_rate: int = TARGET_SAMPLE_RATE,
    device: Optional[str] = None,
) -> torch.Tensor:
    """Returns (seq_len, 1024) frozen frontend hidden states for one clip, on CPU."""
    device = device or DEVICE
    feature_extractor, frontend = load_frontend(device=device)

    audio_array = ensure_target_sample_rate(audio_array, sampling_rate)

    inputs = feature_extractor(
        audio_array,
        sampling_rate=TARGET_SAMPLE_RATE,
        return_tensors="pt",
    )
    input_values = inputs.input_values.to(device)

    with torch.no_grad():
        outputs = frontend(input_values)

    hidden_states = outputs.last_hidden_state.squeeze(0)
    return hidden_states.cpu()



In [82]:
import inspect, model_pipeline

print(inspect.getsource(model_pipeline.load_frontend))

def load_frontend(
    model_name: str = FRONTEND_MODEL_NAME,
    device: Optional[str] = None,
) -> Tuple[Any, nn.Module]:
    """
    Load and freeze the wav2vec2-XLS-R-300M frontend. Cached after first call.

    Returns (feature_extractor, frozen_frontend). Verified to freeze to
    0 trainable parameters.
    """
    from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model

    device = device or DEVICE

    cache_key = (model_name, str(device))
    if cache_key in _FRONTEND_CACHE:
        return _FRONTEND_CACHE[cache_key]

    feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)
    frontend = Wav2Vec2Model.from_pretrained(model_name)  #"Load an already-trained version of this model."

    for param in frontend.parameters(): #Go through all the parameters of wav2vec2 and don't calculate/update gradients for them.
        param.requires_grad = False
    frontend.eval()  #puts wav2vec2 into evaluation mode.
    frontend = frontend.to(device)
    _FRONTEN

In [83]:
from pathlib import Path

root = Path("/content/indic-fake-voice-detection")

for name in ["train.py", "model_pipeline.py"]:
    path = root / name
    if path.exists():
        print(f"{name}:")
        for i, line in enumerate(path.read_text().splitlines(), 1):
            if any(x in line.lower() for x in [
                "extract_features", "sampling_rate",
                "feature_extractor", "normalize"
            ]):
                print(f"{i}: {line.strip()}")

model_pipeline.py:
73: normalize_language_hint,
498: Returns (feature_extractor, frozen_frontend). Verified to freeze to
509: feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)
516: _FRONTEND_CACHE[cache_key] = (feature_extractor, frontend)
518: return feature_extractor, frontend
534: def extract_features(
536: sampling_rate: int = TARGET_SAMPLE_RATE,
541: feature_extractor, frontend = load_frontend(device=device)
543: audio_array = ensure_target_sample_rate(audio_array, sampling_rate)
545: inputs = feature_extractor(
547: sampling_rate=TARGET_SAMPLE_RATE,
569: feature_extractor, _ = load_frontend()
570: return bool(getattr(feature_extractor, "return_attention_mask", False))
573: def extract_features_batch(
575: sampling_rates: Optional[Sequence[int]] = None,
582: Numerically equivalent to calling extract_features() per clip, not an
597: feature_extractor, frontend = load_frontend(device=device)
599: if sampling_rates is None:
600: sampling_rates = [TARGET_SAMPLE_

In [84]:
from pathlib import Path

root = Path("/content/indic-fake-voice-detection")

for p in root.rglob("*.py"):
    if any(x in str(p) for x in [".git", "__pycache__"]):
        continue
    if "train" in p.name.lower() or "feature" in p.name.lower():
        print(p.relative_to(root))

In [85]:
from pathlib import Path

root = Path("/content/indic-fake-voice-detection")

for p in sorted(root.iterdir()):
    print(("/" if p.is_dir() else "   ") + p.name)

/.git
   .gitignore
/.pytest_cache
/.streamlit
   Evaluation.gitkeep
/Evidence report
   README.md
/__pycache__
   app.py
   call_analysis.py
/checkpoints
   codec_robustness.py
   confusion_matrix_Assamese.png
   confusion_matrix_Bengali.png
   confusion_matrix_Bodo.png
   confusion_matrix_Dogri.png
   confusion_matrix_English.png
   confusion_matrix_Gujarati.png
   confusion_matrix_Hindi.png
   confusion_matrix_Kannada.png
   confusion_matrix_Malayalam.png
   confusion_matrix_Manipuri.png
   confusion_matrix_Marathi.png
   confusion_matrix_Nepali.png
   confusion_matrix_Odia.png
   confusion_matrix_Sanskrit.png
   confusion_matrix_Tamil.png
   confusion_matrix_Telugu.png
/data_pipeline
/docs
   evaluate_test_split.py
/evaluation
   evidence_report.py
   i18n.py
/kaggle
   model_pipeline.py
/model_pipeline_artifacts
   requirements-kaggle.txt
   requirements-training.txt
   requirements.txt
   run_validation_ablations.py
   test_analyze_bytes_e2e.py
/tests
   ui_kit.py
   voice_guard_

In [86]:
from pathlib import Path

root = Path("/content/indic-fake-voice-detection")

for folder in ["kaggle", "data_pipeline"]:
    print(f"\n{folder}/")
    for p in sorted((root / folder).iterdir()):
        print(" ", p.name)


kaggle/
  voice_guard_demo.ipynb
  voice_guard_v2_training.ipynb

data_pipeline/
  .gitkeep
  __pycache__
  build_group_safe_split.py
  degraded
  export_hf_metadata.py
  leakage_audit.py
  split_ids.json
  split_protocol.py


In [87]:
import json
from pathlib import Path

p = Path("/content/indic-fake-voice-detection/kaggle/voice_guard_v2_training.ipynb")
nb = json.loads(p.read_text())

for i, cell in enumerate(nb["cells"]):
    if cell["cell_type"] == "code":
        code = "".join(cell["source"])
        keywords = ["Wav2Vec2FeatureExtractor", "sampling_rate", "normalize", "extract_features"]
        if any(k.lower() in code.lower() for k in keywords):
            print(f"\n--- CELL {i} ---")
            print(code[:1800])

In [88]:
import json
from pathlib import Path

p = Path("/content/indic-fake-voice-detection/kaggle/voice_guard_v2_training.ipynb")
nb = json.loads(p.read_text())

print("Total cells:", len(nb["cells"]))
for i, c in enumerate(nb["cells"]):
    text = "".join(c.get("source", []))
    print(i, c["cell_type"], text[:100].replace("\n", " "))

Total cells: 8
0 markdown # Voice Guard v2 training  This notebook creates a transcript/prompt-disjoint v2 protocol, builds fr
1 code import os import subprocess import sys from pathlib import Path  REPO_URL = "https://github.com/Shru
2 markdown ## 1. Export metadata and freeze v2  This pass removes the audio column before streaming, so it read
3 code METADATA = WORK_DIR / "metadata_v2.jsonl" SPLIT = WORK_DIR / "split_ids_v2.json" subprocess.run([sys
4 markdown ## 2. Validation-only ablations  `smoke` is the safe default and is not reportable. After it complet
5 code RUN_MODE = os.environ.get("VOICE_GUARD_TRAIN_MODE", "smoke").lower() if RUN_MODE not in {"smoke", "f
6 markdown ## 3. Final-test gate  Do not evaluate test during model selection. First review the full validation
7 code RUN_FINAL_TEST = os.environ.get("VOICE_GUARD_RUN_FINAL_TEST", "0").strip() == "1" LOCKED_CHECKPOINT_


In [89]:
import json
from pathlib import Path

p = Path("/content/indic-fake-voice-detection/kaggle/voice_guard_v2_training.ipynb")
nb = json.loads(p.read_text())

print("".join(nb["cells"][5]["source"])[:3500])

RUN_MODE = os.environ.get("VOICE_GUARD_TRAIN_MODE", "smoke").lower()
if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("VOICE_GUARD_TRAIN_MODE must be 'smoke' or 'full'.")
epochs = 2 if RUN_MODE == "smoke" else 15
max_clips = 400 if RUN_MODE == "smoke" else None
CACHE = WORK_DIR / ("feature_cache_smoke" if RUN_MODE == "smoke" else "feature_cache_full")
CHECKPOINTS = WORK_DIR / ("experiments_smoke" if RUN_MODE == "smoke" else "experiments_full")
RESULTS = WORK_DIR / f"validation_ablation_results_{RUN_MODE}.json"
command = [sys.executable, str(REPO_DIR / "run_validation_ablations.py"), "--split-file", str(SPLIT), "--metadata-file", str(METADATA), "--cache-dir", str(CACHE), "--checkpoint-root", str(CHECKPOINTS), "--output", str(RESULTS), "--epochs", str(epochs)]
if max_clips is not None:
    command += ["--max-clips-per-split", str(max_clips)]
print("Mode:", RUN_MODE, "| Results:", RESULTS)
subprocess.run(command, cwd=REPO_DIR, check=True)



In [90]:
from pathlib import Path

p = Path("/content/indic-fake-voice-detection/run_validation_ablations.py")
print("File exists:", p.exists())
print("File size:", p.stat().st_size if p.exists() else "Missing")

File exists: True
File size: 4575


In [91]:
from pathlib import Path

p = Path("/content/indic-fake-voice-detection/run_validation_ablations.py")
for i, line in enumerate(p.read_text().splitlines(), 1):
    if any(x in line.lower() for x in [
        "extract_features", "feature_cache", "sampling_rate",
        "wav2vec", "train", "normalize"
    ]):
        print(f"{i}: {line.strip()}")

20: def candidate_configs(base: mp.TrainConfig) -> Dict[str, mp.TrainConfig]:
54: parser.add_argument("--cache-dir", default="feature_cache_v2")
71: base = mp.TrainConfig(num_epochs=args.epochs, batch_size=args.batch_size)
78: summary = mp.run_step6_training(


In [92]:
import inspect, model_pipeline

print(inspect.getsource(model_pipeline.run_step6_training)[:3500])

def run_step6_training(
    config: Optional[TrainConfig] = None,
    split_file: str = SPLIT_FILE,
    metadata_file: str = METADATA_FILE,
    cache_dir: str = FEATURE_CACHE_DIR,
    checkpoint_dir: str = CHECKPOINT_DIR,
    shard_size: int = 500,
    store_dtype: torch.dtype = torch.float16,
    train_in_memory: bool = False,
    window_shards: int = 4,
    max_clips_per_split: Optional[int] = None,
    skip_extraction: bool = False,
    extract_batch_size: int = 8,
    max_failure_pct: float = 1.0,
) -> Dict[str, Any]:
    """
    Full Step 6 run: guards -> feature cache -> train -> calibrate.

    Stages:
        1. assert_split_integrity()   — refuse exact-ID and proxy-group leakage
        2. verify_frontend_frozen()   — assert 0 trainable frontend params
        3. load frozen split + label lookup
        4. build_feature_caches_streaming() for train + validation in ONE pass
           (skipped if already complete, or if skip_extraction=True)
        5. train_model() over the sh

In [93]:
import inspect, model_pipeline

print(inspect.getsource(
    model_pipeline.build_feature_caches_streaming
)[:2500])

def build_feature_caches_streaming(
    targets: Dict[str, set],
    label_lookup: Dict[str, int],
    language_lookup: Optional[Dict[str, str]] = None,
    cache_dir: str = FEATURE_CACHE_DIR,
    shard_size: int = 500,
    store_dtype: torch.dtype = torch.float16,
    device: Optional[str] = None,
    resume: bool = True,
    progress_every: int = 100,
    max_clips_per_split: Optional[int] = None,
    extract_batch_size: int = 8,
    batch_sort_factor: int = 4,
    # 120 s of padded audio per forward pass. Self-attention is quadratic in
    # sequence length, so this is what stops 8 x 30 s clips (T=1500 padded frames,
    # ~1 GB of transient attention per layer) from landing in one batch.
    max_batch_samples: int = 120 * TARGET_SAMPLE_RATE,
) -> Dict[str, Dict[str, Any]]:
    """
    Single-pass, memory-bounded feature cache builder.

    Opens the HF stream ONCE and builds caches for every split in `targets`
    simultaneously (e.g. {"train": train_ids, "validation": val_ids}), b

In [94]:
import inspect, model_pipeline

print(inspect.getsource(model_pipeline.extract_features_batch)[:2500])

def extract_features_batch(
    audio_arrays: Sequence[np.ndarray],
    sampling_rates: Optional[Sequence[int]] = None,
    device: Optional[str] = None,
) -> List[torch.Tensor]:
    """
    Batched frozen-frontend extraction. Returns one (seq_len_i, 1024) CPU tensor
    per input, each trimmed to its own true length.

    Numerically equivalent to calling extract_features() per clip, not an
    approximation of it, because:

      * the feature extractor pads and returns an attention_mask, and applies its
        zero-mean/unit-variance normalization per sequence over the unpadded
        length (not over the padded tensor);
      * the mask is passed to the model, so padded frames are excluded from
        self-attention;
      * each output is sliced back to its own frame count via the model's
        conv-stack length formula, so no padded frame is ever stored.

    Falls back to per-clip extraction if the feature extractor has no attention
    mask — correctness first.
    """
   

In [95]:
import numpy as np
import model_pipeline

audio = np.random.default_rng(42).normal(
    0, 0.05, 16000 * 5
).astype(np.float32)

single = model_pipeline.extract_features(audio, 16000)
batch = model_pipeline.extract_features_batch([audio], [16000])[0]

print("Single shape:", single.shape)
print("Batch shape:", batch.shape)
print("Maximum difference:", (single - batch).abs().max().item())

Single shape: torch.Size([249, 1024])
Batch shape: torch.Size([249, 1024])
Maximum difference: 0.0


In [96]:
from pathlib import Path
import numpy as np
import soundfile as sf
import model_pipeline

p = Path("/content/WhatsApp Audio 2026-11-16 at 6.43.34 PM.aac")

audio, sr = model_pipeline.decode_audio_bytes(p.read_bytes())
audio = np.asarray(audio, dtype=np.float32).squeeze()

print("Sample rate:", sr)
print("Duration:", round(len(audio) / sr, 2), "seconds")
print("Shape:", audio.shape)
print("RMS:", round(float(np.sqrt(np.mean(audio ** 2))), 6))
print("Peak:", round(float(np.max(np.abs(audio))), 6))
print("Minimum:", round(float(audio.min()), 6))
print("Maximum:", round(float(audio.max()), 6))
print("Finite values:", bool(np.isfinite(audio).all()))

Sample rate: 48000
Duration: 26.75 seconds
Shape: (1284096,)
RMS: 0.238438
Peak: 0.90538
Minimum: -0.884445
Maximum: 0.90538
Finite values: True


In [97]:
from pathlib import Path

root = Path("/content/indic-fake-voice-detection")

for p in root.rglob("*manifest*"):
    if p.is_file():
        print(p.relative_to(root))

data_pipeline/degraded/musan_10db_manifest (1).jsonl
data_pipeline/degraded/opus_16kbps_manifest.jsonl


In [98]:
from pathlib import Path

root = Path("/content/indic-fake-voice-detection")

for p in root.rglob("*"):
    if p.is_dir() and "feature_cache" in p.name.lower():
        print(p.relative_to(root))

In [99]:
import json
from pathlib import Path

p = Path("/content/indic-fake-voice-detection/kaggle/voice_guard_v2_training.ipynb")
nb = json.loads(p.read_text())

for i in [1, 3, 5, 7]:
    print(f"\n--- CELL {i} ---")
    print("".join(nb["cells"][i]["source"])[:1200])


--- CELL 1 ---
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Shruti-sivaramakrishnan/indic-fake-voice-detection.git"
REPO_DIR = Path("/kaggle/working/indic-fake-voice-detection")
WORK_DIR = Path("/kaggle/working/voice_guard_v2")
VOICE_GUARD_REF = os.environ.get("VOICE_GUARD_REF", "main").strip() or "main"
WORK_DIR.mkdir(parents=True, exist_ok=True)

if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--filter=blob:none", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", VOICE_GUARD_REF], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements-training.txt")], check=True)
RESOLVED_SHA = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print("Checked out:", RES

In [100]:
from pathlib import Path

for folder in [
    "/kaggle/working/voice_guard_v2",
    "/content/voice_guard_v2",
    "/content/indic-fake-voice-detection/checkpoints",
]:
    p = Path(folder)
    print(folder, "EXISTS" if p.exists() else "NOT FOUND")

/kaggle/working/voice_guard_v2 NOT FOUND
/content/voice_guard_v2 NOT FOUND
/content/indic-fake-voice-detection/checkpoints EXISTS


In [101]:
import torch

path = "/content/indic-fake-voice-detection/checkpoints/best_model.pt"
ckpt = torch.load(path, map_location="cpu", weights_only=False)

print("Checkpoint keys:", list(ckpt.keys()))

Checkpoint keys: ['model_state_dict', 'optimizer_state_dict', 'epoch', 'loss_history', 'metrics', 'config', 'frontend_model_name', 'best_eer', 'history']


In [102]:
print("Epoch:", ckpt.get("epoch"))
print("Best EER:", ckpt.get("best_eer"))
print("Frontend:", ckpt.get("frontend_model_name"))
print("Config:", ckpt.get("config"))

Epoch: 11
Best EER: 0.00996784565916397
Frontend: facebook/wav2vec2-xls-r-300m
Config: {'hidden_dim': 256, 'num_gat_layers': 2, 'dropout': 0.3, 'learning_rate': 0.0001, 'weight_decay': 0.01, 'num_epochs': 15, 'batch_size': 16, 'shuffle_train': True, 'grad_clip': 0.0, 'seed': 42}


In [103]:
print("Metrics:")
print(ckpt.get("metrics"))

print("\nHistory type:", type(ckpt.get("history")).__name__)
print("History:")
print(ckpt.get("history"))

Metrics:
{'eer': 0.00996784565916397, 'threshold': 2.1031923294067383, 'loss': 0.04063492268323898}

History type: list
History:
[{'epoch': 1, 'train_loss': 0.3405411530521899, 'val_loss': 0.9340847134590149, 'val_eer': 0.22057877813504823}, {'epoch': 2, 'train_loss': 0.2955202936709909, 'val_loss': 0.6674821376800537, 'val_eer': 0.16270096463022504}, {'epoch': 3, 'train_loss': 0.19118477386318392, 'val_loss': 0.6033693552017212, 'val_eer': 0.08713826366559486}, {'epoch': 4, 'train_loss': 0.09860777470973467, 'val_loss': 0.164616659283638, 'val_eer': 0.041479099678456574}, {'epoch': 5, 'train_loss': 0.06086201902004569, 'val_loss': 0.2951403856277466, 'val_eer': 0.03504823151125401}, {'epoch': 6, 'train_loss': 0.054444159730566835, 'val_loss': 0.09234293550252914, 'val_eer': 0.0189710610932476}, {'epoch': 7, 'train_loss': 0.05615249359983267, 'val_loss': 0.11940348893404007, 'val_eer': 0.030225080385852116}, {'epoch': 8, 'train_loss': 0.042109581662691103, 'val_loss': 0.048063576221466

In [104]:
from pathlib import Path

root = Path("/content/indic-fake-voice-detection")

for pattern in ["*metadata*", "*manifest*", "*dataset*", "*split*"]:
    print(f"\n{pattern}")
    for p in root.rglob(pattern):
        if p.is_file():
            print(p.relative_to(root))


*metadata*
data_pipeline/export_hf_metadata.py
model_pipeline_artifacts/split_metadata_verified.json
data_pipeline/__pycache__/export_hf_metadata.cpython-313.pyc

*manifest*
data_pipeline/degraded/musan_10db_manifest (1).jsonl
data_pipeline/degraded/opus_16kbps_manifest.jsonl

*dataset*

*split*
evaluate_test_split.py
data_pipeline/split_protocol.py
data_pipeline/build_group_safe_split.py
data_pipeline/split_ids.json
tests/test_split_protocol.py
model_pipeline_artifacts/split_metadata_verified.json
tests/__pycache__/test_split_protocol.cpython-313-pytest-8.4.2.pyc
data_pipeline/__pycache__/split_protocol.cpython-313.pyc
data_pipeline/__pycache__/build_group_safe_split.cpython-313.pyc


In [105]:
from pathlib import Path
import json

root = Path("/content/indic-fake-voice-detection")

script = root / "data_pipeline/export_hf_metadata.py"
print("=== METADATA EXPORT SCRIPT ===")
print("\n".join(script.read_text().splitlines()[:100]))

meta = root / "model_pipeline_artifacts/split_metadata_verified.json"
print("\n=== VERIFIED SPLIT METADATA ===")
data = json.loads(meta.read_text())
print(json.dumps(data, indent=2)[:5000])

=== METADATA EXPORT SCRIPT ===
"""Export label/language/prompt metadata without downloading or decoding audio."""

from __future__ import annotations

import argparse
import hashlib
import json
import re
import unicodedata
from collections import Counter
from pathlib import Path
from typing import Any, Dict, Iterable, Optional


DEFAULT_DATASET = "SherryT997/IndicTTS-Deepfake-Challenge-Data"
DEFAULT_SPLIT = "train"
EXPECTED_TOTAL = 31_102


def normalize_text(value: Any) -> str:
    text = unicodedata.normalize("NFKC", str(value)).casefold().strip()
    return re.sub(r"\s+", " ", text)


def prompt_id(language: str, text: str) -> str:
    normalized = f"{normalize_text(language)}\0{normalize_text(text)}"
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


def id_family(clip_id: str) -> str:
    return re.sub(r"[_-]?\d+$", "", clip_id.strip()).casefold()


def metadata_record(example: Dict[str, Any]) -> Dict[str, Any]:
    clip_id = str(example.get("id", "")).strip()
   

In [106]:
import json
from collections import Counter, defaultdict
from pathlib import Path

path = Path("/content/indic-fake-voice-detection/model_pipeline_artifacts/split_metadata_verified.json")
splits = json.loads(path.read_text())

for split_name, records in splits.items():
    counts = Counter()
    languages = defaultdict(Counter)

    for clip_id, language, label in records:
        counts[label] += 1
        languages[language][label] += 1

    print(f"\n{split_name.upper()}: {len(records)} clips")
    print(f"Real: {counts[0]} | Fake: {counts[1]}")

    for language, labels in sorted(languages.items()):
        print(f"  {language}: Real={labels[0]}, Fake={labels[1]}")


TRAIN: 24881 clips
Real: 12445 | Fake: 12436
  Assamese: Real=801, Fake=799
  Bengali: Real=718, Fake=722
  Bodo: Real=802, Fake=798
  Dogri: Real=800, Fake=800
  English: Real=800, Fake=800
  Gujarati: Real=623, Fake=627
  Hindi: Real=800, Fake=800
  Kannada: Real=799, Fake=801
  Malayalam: Real=803, Fake=797
  Manipuri: Real=800, Fake=800
  Marathi: Real=802, Fake=798
  Nepali: Real=799, Fake=801
  Odia: Real=729, Fake=730
  Sanskrit: Real=802, Fake=798
  Tamil: Real=767, Fake=765
  Telugu: Real=800, Fake=800

VALIDATION: 3110 clips
Real: 1555 | Fake: 1555
  Assamese: Real=100, Fake=100
  Bengali: Real=90, Fake=91
  Bodo: Real=100, Fake=100
  Dogri: Real=100, Fake=100
  English: Real=100, Fake=100
  Gujarati: Real=78, Fake=78
  Hindi: Real=100, Fake=100
  Kannada: Real=100, Fake=100
  Malayalam: Real=100, Fake=99
  Manipuri: Real=100, Fake=100
  Marathi: Real=100, Fake=99
  Nepali: Real=100, Fake=100
  Odia: Real=91, Fake=92
  Sanskrit: Real=100, Fake=100
  Tamil: Real=96, Fake=96
 

In [107]:
from pathlib import Path

root = Path("/content/indic-fake-voice-detection")

for filename in [
    "data_pipeline/split_protocol.py",
    "data_pipeline/build_group_safe_split.py",
]:
    path = root / filename
    print(f"\n===== {filename} =====")

    for number, line in enumerate(path.read_text().splitlines(), 1):
        if any(term in line.lower() for term in [
            "prompt_id", "source_family", "group", "split", "leakage"
        ]):
            print(f"{number}: {line[:180]}")


===== data_pipeline/split_protocol.py =====
1: """Verification helpers for reportable Voice Guard split artifacts."""
12: PROTOCOL_VERSION = "v2-group-disjoint"
23: def default_audit_path(split_file: Path) -> Path:
24:     return split_file.with_suffix(".audit.json")
27: def verify_split_audit(
28:     split_file: Path | str,
32:     """Verify that a v2 audit is complete and bound to the exact split bytes."""
33:     split_path = Path(split_file)
34:     audit_path = Path(audit_file) if audit_file else default_audit_path(split_path)
35:     if not split_path.exists():
36:         raise FileNotFoundError(f"Split file not found: {split_path}")
39:             f"Split audit not found: {audit_path}. Generate the split with "
40:             "data_pipeline/build_group_safe_split.py."
46:             f"Unsupported split protocol {audit.get('protocol_version')!r}; "
49:     if audit.get("split_sha256") != sha256_file(split_path):
51:             "Split audit SHA-256 does not match the split 

In [109]:
from pathlib import Path

root = Path("/content/indic-fake-voice-detection")

for folder in ["data_pipeline", "model_pipeline_artifacts"]:
    print(f"\n=== {folder} ===")
    for p in sorted((root / folder).glob("*")):
        if p.is_file() and any(
            word in p.name.lower()
            for word in ["audit", "split", "metadata"]
        ):
            print(p.name, "|", p.stat().st_size, "bytes")


=== data_pipeline ===
build_group_safe_split.py | 13149 bytes
export_hf_metadata.py | 4639 bytes
leakage_audit.py | 8383 bytes
split_ids.json | 790655 bytes
split_protocol.py | 4270 bytes

=== model_pipeline_artifacts ===
split_metadata_verified.json | 1148119 bytes


In [110]:
from pathlib import Path
import json

root = Path("/content/indic-fake-voice-detection")

print("=== Expected audit file ===")
audit = root / "data_pipeline/split_ids.audit.json"
print("Exists:", audit.exists())

print("\n=== Split file structure ===")
split = json.loads((root / "data_pipeline/split_ids.json").read_text())
print("Keys:", list(split.keys()))

print("\n=== Leakage audit requirements ===")
script = root / "data_pipeline/leakage_audit.py"
for i, line in enumerate(script.read_text().splitlines(), 1):
    if i <= 120:
        print(f"{i}: {line}")

=== Expected audit file ===
Exists: False

=== Split file structure ===
Keys: ['train', 'validation', 'test']

=== Leakage audit requirements ===
1: """Audit exact IDs, source-family proxies, and degraded-audio lineage.
2: 
3: The original audit checked only exact clip IDs. That is necessary but not
4: sufficient: clips with the same source/emotion/speaker-like ID prefix are spread
5: across all three v1 splits. This audit makes that limitation visible and fails by
6: default until a group-disjoint v2 split is supplied.
7: """
8: 
9: from __future__ import annotations
10: 
11: import argparse
12: import json
13: import re
14: from collections import Counter
15: from pathlib import Path
16: from typing import Dict, Iterable, List
17: 
18: try:
19:     from data_pipeline.split_protocol import verify_split_audit
20: except ModuleNotFoundError:  # direct `python data_pipeline/leakage_audit.py`
21:     from split_protocol import verify_split_audit
22: 
23: 
24: REPO_ROOT = Path(__file__).re

In [111]:
from pathlib import Path
import json

root = Path("/content/indic-fake-voice-detection")

notebook = root / "kaggle/voice_guard_v2_training.ipynb"
data = json.loads(notebook.read_text())

for i, cell in enumerate(data["cells"]):
    source = "".join(cell.get("source", []))
    if any(term in source.lower() for term in [
        "split_ids", "metadata_v2", "split_metadata_verified",
        "group_columns", "run_step6_training"
    ]):
        print(f"\n=== CELL {i} ===")
        print(source[:2500])


=== CELL 3 ===
METADATA = WORK_DIR / "metadata_v2.jsonl"
SPLIT = WORK_DIR / "split_ids_v2.json"
subprocess.run([sys.executable, str(REPO_DIR / "data_pipeline/export_hf_metadata.py"), "--output", str(METADATA)], check=True)
subprocess.run([sys.executable, str(REPO_DIR / "data_pipeline/build_group_safe_split.py"), str(METADATA), "--group-columns", "prompt_id", "--output", str(SPLIT)], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, str(REPO_DIR / "data_pipeline/leakage_audit.py"), "--split-file", str(SPLIT)], cwd=REPO_DIR, check=True)



In [112]:
from pathlib import Path

root = Path("/content/indic-fake-voice-detection")

for p in root.rglob("*"):
    if p.is_file() and (
        "split_ids_v2" in p.name
        or "audit.json" in p.name
        or "metadata_v2" in p.name
    ):
        print(p.relative_to(root), "|", p.stat().st_size, "bytes")

evaluation/threshold_orientation_audit.json | 3311 bytes


In [113]:
import json
from pathlib import Path

path = Path(
    "/content/indic-fake-voice-detection/evaluation/threshold_orientation_audit.json"
)

data = json.loads(path.read_text())
print(json.dumps(data, indent=2))

{
  "finding": "Historical robustness classification artifacts used the validation bonafide-score threshold as a fake-logit threshold without negating it.",
  "validation_eer_threshold_bonafide_score": 2.1031923294067383,
  "validation_eer_threshold_fake_logit": -2.1031923294067383,
  "selection_warning": "The corrected validation threshold is reported for consistency; do not choose among these operating points using test accuracy.",
  "datasets": [
    {
      "file": "evaluation/clean/frozen_test_predictions_with_language.csv",
      "n": 3111,
      "operating_points": {
        "raw_logit_zero": {
          "fake_logit_threshold": 0.0,
          "accuracy": 0.9839279974284796,
          "tn": 1553,
          "fp": 5,
          "fn": 45,
          "tp": 1508
        },
        "historical_wrong_sign": {
          "fake_logit_threshold": 2.1031923294067383,
          "warning": "Bonafide-score threshold was used without negating it.",
          "accuracy": 0.9633558341369335,
       

In [114]:
from pathlib import Path

root = Path("/content/indic-fake-voice-detection")
terms = [
    "validation_eer_threshold_bonafide_score",
    "validation_eer_threshold_fake_logit",
    "fake_logit_threshold",
]

for path in root.rglob("*.py"):
    if any(part in str(path) for part in [".git", "__pycache__"]):
        continue

    lines = path.read_text(errors="ignore").splitlines()
    matches = [
        (i, line.strip())
        for i, line in enumerate(lines, 1)
        if any(term in line for term in terms)
    ]

    if matches:
        print(f"\n=== {path.relative_to(root)} ===")
        for number, line in matches:
            print(f"{number}: {line[:200]}")


=== model_pipeline.py ===
3137: "validation_eer_threshold_bonafide_score": (
3140: "validation_eer_threshold_fake_logit": (

=== evaluation/audit_threshold_orientation.py ===
40: "fake_logit_threshold": 0.0,
44: "fake_logit_threshold": bonafide_threshold,
49: "fake_logit_threshold": -bonafide_threshold,
91: "validation_eer_threshold_bonafide_score": bonafide_threshold,
92: "validation_eer_threshold_fake_logit": -bonafide_threshold,


In [115]:
from pathlib import Path

path = Path("/content/indic-fake-voice-detection/model_pipeline.py")
lines = path.read_text().splitlines()

for start, end in [(3095, 3160)]:
    for i in range(start, min(end, len(lines))):
        print(f"{i + 1}: {lines[i]}")

3096:     )
3097: 
3098:     detected_language = normalize_language_hint(language_hint)
3099:     language_source = "user_declared" if detected_language != "unknown" else "none"
3100: 
3101:     return AnalysisResult(
3102:         verdict=verdict,
3103:         confidence=float(confidence),
3104:         detected_language=detected_language,
3105:         language_eer=None,
3106:         raw_score=float(raw_logit),
3107:         audio_hash=audio_hash,
3108:         metadata={
3109:             "p_fake_calibrated": float(p_fake),
3110:             "temperature": bundle.temperature,
3111:             "thresholds": {
3112:                 "real_max_p_fake": VERDICT_REAL_MAX_P_FAKE,
3113:                 "fake_min_p_fake": VERDICT_FAKE_MIN_P_FAKE,
3114:             },
3115:             "decision_policy_id": DEFAULT_DECISION_POLICY.policy_id,
3116:             "decision_policy_provenance": DEFAULT_DECISION_POLICY.provenance,
3117:             "duration_seconds": quality["duration_seconds"],

In [116]:
from pathlib import Path

path = Path("/content/indic-fake-voice-detection/model_pipeline.py")
lines = path.read_text().splitlines()

for i, line in enumerate(lines):
    if "def verdict_from_p_fake" in line:
        start = max(0, i - 8)
        end = min(len(lines), i + 35)
        for j in range(start, end):
            print(f"{j + 1}: {lines[j]}")
        break

2851: # Compatibility aliases for callers that imported these names. The canonical
2852: # policy and its provenance now live in voice_guard_core.py. The v1 values are
2853: # preserved until a v2 validation run can derive replacement operating points;
2854: # the already-observed v1 test set must never be used to tune them.
2855: VERDICT_REAL_MAX_P_FAKE = DEFAULT_DECISION_POLICY.real_max_p_fake
2856: VERDICT_FAKE_MIN_P_FAKE = DEFAULT_DECISION_POLICY.fake_min_p_fake
2857: 
2858: 
2859: def verdict_from_p_fake(
2860:     p_fake: float,
2861:     real_max: float = VERDICT_REAL_MAX_P_FAKE,
2862:     fake_min: float = VERDICT_FAKE_MIN_P_FAKE,
2863: ) -> str:
2864:     if (
2865:         real_max == DEFAULT_DECISION_POLICY.real_max_p_fake
2866:         and fake_min == DEFAULT_DECISION_POLICY.fake_min_p_fake
2867:     ):
2868:         return verdict_from_probability(p_fake)
2869:     from voice_guard_core import DecisionPolicy
2870: 
2871:     return verdict_from_probability(
2872:         p

In [117]:
from voice_guard_core import DEFAULT_DECISION_POLICY

print("Policy ID:", DEFAULT_DECISION_POLICY.policy_id)
print("Real maximum:", DEFAULT_DECISION_POLICY.real_max_p_fake)
print("Fake minimum:", DEFAULT_DECISION_POLICY.fake_min_p_fake)
print("Provenance:", DEFAULT_DECISION_POLICY.provenance)

Policy ID: legacy-symmetric-abstention-v1
Real maximum: 0.35
Fake minimum: 0.65
Provenance: Legacy symmetric abstention band; not selected on the frozen test set. Replace only from validation data under the v2 protocol.


In [118]:
from pathlib import Path

files = list(Path("/content").glob("*.mp4"))

for f in files:
    print(f.name, "|", round(f.stat().st_size / 1024, 2), "KB")

WhatsApp Audio 2026-11-16 at 7.27.32 PM.mp4 | 509.05 KB
WhatsApp Audio 2026-11-16 at 6.35.58 PM.mp4 | 806.47 KB


In [119]:
from pathlib import Path
from call_analysis import analyze_call

audio_path = Path("/content/WhatsApp Audio 2026-11-16 at 7.27.32 PM.mp4")
audio_bytes = audio_path.read_bytes()

report = analyze_call(
    audio_bytes,
    progress_callback=lambda p: print(f"Progress: {p:.0%}")
)

print("\n===== CALL ANALYSIS RESULT =====")
print("Duration:", report.duration_seconds)
print("Active audio:", report.speech_duration_seconds)
print("Windows analysed:", report.windows_analysed)
print("Flagged windows:", report.flagged_windows)
print("Flagged share:", f"{report.flagged_share:.1%}")
print("Highest fake probability:", report.highest_p_fake)
print("Highest score timestamp:", report.highest_score_timestamp)

print("\n===== INDIVIDUAL WINDOWS =====")
for window in report.window_scores:
    print(
        f"{window.start_seconds:.2f}s - "
        f"{window.end_seconds:.2f}s | "
        f"P(fake): {window.p_fake:.4f} | "
        f"Verdict: {window.verdict}"
    )

Progress: 12%
Progress: 25%
Progress: 38%
Progress: 50%
Progress: 62%
Progress: 75%
Progress: 88%
Progress: 100%

===== CALL ANALYSIS RESULT =====
Duration: 31.872
Active audio: 27.330000000000002
Windows analysed: 8
Flagged windows: 1
Flagged share: 12.5%
Highest fake probability: 0.8764105280479437
Highest score timestamp: 24.95

===== INDIVIDUAL WINDOWS =====
1.91s - 7.91s | P(fake): 0.0020 | Verdict: real
4.91s - 10.91s | P(fake): 0.0070 | Verdict: real
7.91s - 11.36s | P(fake): 0.0023 | Verdict: real
11.87s - 14.87s | P(fake): 0.1227 | Verdict: real
17.03s - 22.75s | P(fake): 0.0123 | Verdict: real
20.03s - 22.75s | P(fake): 0.3182 | Verdict: real
24.95s - 27.68s | P(fake): 0.8764 | Verdict: fake
27.98s - 31.86s | P(fake): 0.0056 | Verdict: real


In [120]:
import numpy as np
from pathlib import Path
import model_pipeline as mp

audio_path = Path("/content/WhatsApp Audio 2026-11-16 at 7.27.32 PM.mp4")

audio_bytes = audio_path.read_bytes()
waveform, sr = mp.decode_audio_bytes(audio_bytes)
waveform = mp.ensure_target_sample_rate(waveform, sr)
waveform = np.asarray(waveform, dtype=np.float32).reshape(-1)

bundle = mp.load_inference_bundle()

segments = [
    ("Preceding section", 20.03, 22.75),
    ("Flagged section", 24.95, 27.68),
    ("Following section", 27.98, 31.86),
]

for name, start, end in segments:
    clip = waveform[int(start * 16000):int(end * 16000)]

    raw_logit, p_fake = mp.score_waveform(
        clip,
        sample_rate=16000,
        bundle=bundle
    )

    verdict = mp.verdict_from_p_fake(p_fake)

    print(
        f"{name}: {start:.2f}-{end:.2f}s | "
        f"Duration: {len(clip)/16000:.2f}s | "
        f"Raw logit: {raw_logit:.4f} | "
        f"P(fake): {p_fake:.4f} | "
        f"Verdict: {verdict}"
    )

Preceding section: 20.03-22.75s | Duration: 2.72s | Raw logit: -0.9606 | P(fake): 0.3181 | Verdict: real
Flagged section: 24.95-27.68s | Duration: 2.73s | Raw logit: 2.4751 | P(fake): 0.8770 | Verdict: fake
Following section: 27.98-31.86s | Duration: 3.88s | Raw logit: -6.7312 | P(fake): 0.0048 | Verdict: real


In [121]:
segments = [
    ("Flagged only", 24.95, 27.68),
    ("With 1.5s context", 23.45, 29.18),
    ("With 3s context", 21.95, 30.68),
    ("Full nearby context", 20.03, 31.86),
]

for name, start, end in segments:
    clip = waveform[int(start * 16000):int(end * 16000)]

    raw_logit, p_fake = mp.score_waveform(
        clip,
        sample_rate=16000,
        bundle=bundle
    )

    verdict = mp.verdict_from_p_fake(p_fake)

    print(
        f"{name} ({start:.2f}-{end:.2f}s) | "
        f"Duration: {len(clip)/16000:.2f}s | "
        f"P(fake): {p_fake:.4f} | "
        f"Verdict: {verdict}"
    )

Flagged only (24.95-27.68s) | Duration: 2.73s | P(fake): 0.8770 | Verdict: fake
With 1.5s context (23.45-29.18s) | Duration: 5.73s | P(fake): 0.0043 | Verdict: real
With 3s context (21.95-30.68s) | Duration: 8.73s | P(fake): 0.0012 | Verdict: real
Full nearby context (20.03-31.86s) | Duration: 11.83s | P(fake): 0.0004 | Verdict: real


In [122]:
import numpy as np

sample_rate = 16000

regions = [
    ("Before flagged segment", 20.03, 24.95),
    ("Flagged segment", 24.95, 27.68),
    ("After flagged segment", 27.68, 31.86),
]

for name, start, end in regions:
    clip = waveform[int(start * sample_rate):int(end * sample_rate)]

    rms = float(np.sqrt(np.mean(clip ** 2)))
    peak = float(np.max(np.abs(clip)))
    duration = len(clip) / sample_rate

    print(
        f"{name}: {start:.2f}-{end:.2f}s | "
        f"Duration={duration:.2f}s | "
        f"RMS={rms:.5f} | "
        f"Peak={peak:.5f}"
    )

Before flagged segment: 20.03-24.95s | Duration=4.92s | RMS=0.06206 | Peak=0.51901
Flagged segment: 24.95-27.68s | Duration=2.73s | RMS=0.06842 | Peak=0.39973
After flagged segment: 27.68-31.86s | Duration=4.18s | RMS=0.07921 | Peak=0.41655


In [123]:
import soundfile as sf
from pathlib import Path

output_dir = Path("/content/voice_inspection")
output_dir.mkdir(exist_ok=True)

segments = [
    ("before_flagged.wav", 20.03, 24.95),
    ("flagged_segment.wav", 24.95, 27.68),
    ("after_flagged.wav", 27.68, 31.86),
]

for filename, start, end in segments:
    clip = waveform[int(start * 16000):int(end * 16000)]
    output_path = output_dir / filename
    sf.write(output_path, clip, 16000)
    print(f"Saved: {output_path}")

Saved: /content/voice_inspection/before_flagged.wav
Saved: /content/voice_inspection/flagged_segment.wav
Saved: /content/voice_inspection/after_flagged.wav


In [124]:
import numpy as np

sample_rate = 16000
window_size = 6.0

starts = [21.0, 22.0, 23.0, 24.0, 24.95, 25.0]

for start in starts:
    end = min(start + window_size, len(waveform) / sample_rate)
    clip = waveform[int(start * sample_rate):int(end * sample_rate)]

    raw_logit, p_fake = mp.score_waveform(
        clip,
        sample_rate=sample_rate,
        bundle=bundle
    )

    verdict = mp.verdict_from_p_fake(p_fake)

    print(
        f"{start:.2f}-{end:.2f}s | "
        f"P(fake): {p_fake:.4f} | "
        f"Verdict: {verdict}"
    )

21.00-27.00s | P(fake): 0.0010 | Verdict: real
22.00-28.00s | P(fake): 0.2992 | Verdict: real
23.00-29.00s | P(fake): 0.3571 | Verdict: uncertain
24.00-30.00s | P(fake): 0.2716 | Verdict: real
24.95-30.95s | P(fake): 0.1001 | Verdict: real
25.00-31.00s | P(fake): 0.0539 | Verdict: real


In [125]:
for item in report.window_scores:
    if item.start_seconds <= 32 and item.end_seconds >= 20:
        print(
            f"{item.start_seconds:.2f}-{item.end_seconds:.2f}s | "
            f"P(fake): {item.p_fake:.4f} | "
            f"Verdict: {item.verdict}"
        )

17.03-22.75s | P(fake): 0.0123 | Verdict: real
20.03-22.75s | P(fake): 0.3182 | Verdict: real
24.95-27.68s | P(fake): 0.8764 | Verdict: fake
27.98-31.86s | P(fake): 0.0056 | Verdict: real


In [126]:
from call_analysis import detect_speech_regions

regions = detect_speech_regions(waveform, sample_rate=16000)

print("Detected speech regions:")
for i, region in enumerate(regions, 1):
    print(
        f"{i}. {region.start_seconds:.2f}s - "
        f"{region.end_seconds:.2f}s | "
        f"Duration: {region.end_seconds - region.start_seconds:.2f}s"
    )

Detected speech regions:
1. 0.02s - 1.16s | Duration: 1.14s
2. 1.91s - 11.36s | Duration: 9.45s
3. 11.87s - 14.87s | Duration: 3.00s
4. 15.39s - 16.48s | Duration: 1.09s
5. 17.03s - 22.75s | Duration: 5.71s
6. 23.82s - 24.16s | Duration: 0.34s
7. 24.95s - 27.68s | Duration: 2.73s
8. 27.98s - 31.86s | Duration: 3.88s


In [127]:
import call_analysis
import inspect

print(inspect.signature(call_analysis.detect_speech_regions))
print(inspect.getsource(call_analysis.detect_speech_regions))

(waveform: 'np.ndarray', sample_rate: 'int', frame_duration: 'float' = 0.025, hop_duration: 'float' = 0.01, merge_gap_seconds: 'float' = 0.3, min_region_seconds: 'float' = 0.15, energy_threshold: 'float | None' = None) -> 'List[SpeechRegion]'
def detect_speech_regions(
    waveform: np.ndarray,
    sample_rate: int,
    frame_duration: float = 0.025,
    hop_duration: float = 0.010,
    merge_gap_seconds: float = 0.30,
    min_region_seconds: float = 0.15,
    energy_threshold: float | None = None,
) -> List[SpeechRegion]:
    """Find speech-active regions using a lightweight adaptive energy gate.

    This is an energy-based detector, not a trained voice activity detector.
    It preserves timestamps relative to the original recording.
    """
    samples = np.asarray(waveform, dtype=np.float32).reshape(-1)

    if sample_rate <= 0:
        raise ValueError("sample_rate must be positive.")
    if samples.size == 0:
        return []
    if not np.isfinite(samples).all():
        raise

In [128]:
from call_analysis import detect_speech_regions

test_regions = detect_speech_regions(
    waveform,
    sample_rate=16000,
    merge_gap_seconds=1.2,
    min_region_seconds=0.15,
)

print("Speech regions with 1.2-second gap merging:")

for i, region in enumerate(test_regions, 1):
    duration = region.end_seconds - region.start_seconds
    print(
        f"{i}. {region.start_seconds:.2f}s - "
        f"{region.end_seconds:.2f}s | Duration: {duration:.2f}s"
    )

Speech regions with 1.2-second gap merging:
1. 0.02s - 31.86s | Duration: 31.84s


In [129]:
from call_analysis import detect_speech_regions

thresholds = [0.3, 0.5, 0.7, 0.9, 1.0]

for gap in thresholds:
    regions = detect_speech_regions(
        waveform,
        sample_rate=16000,
        merge_gap_seconds=gap,
        min_region_seconds=0.15,
    )

    print(f"\nMerge gap: {gap}s | Regions: {len(regions)}")

    for region in regions:
        duration = region.end_seconds - region.start_seconds
        print(
            f"  {region.start_seconds:.2f}-{region.end_seconds:.2f}s "
            f"({duration:.2f}s)"
        )


Merge gap: 0.3s | Regions: 8
  0.02-1.16s (1.14s)
  1.91-11.36s (9.45s)
  11.87-14.87s (3.00s)
  15.39-16.48s (1.09s)
  17.03-22.75s (5.71s)
  23.82-24.16s (0.34s)
  24.95-27.68s (2.73s)
  27.98-31.86s (3.88s)

Merge gap: 0.5s | Regions: 7
  0.02-1.16s (1.14s)
  1.91-11.36s (9.45s)
  11.87-14.87s (3.00s)
  15.39-16.48s (1.09s)
  17.03-22.75s (5.71s)
  23.82-24.16s (0.34s)
  24.95-31.86s (6.91s)

Merge gap: 0.7s | Regions: 4
  0.02-1.16s (1.14s)
  1.91-22.75s (20.84s)
  23.82-24.16s (0.34s)
  24.95-31.86s (6.91s)

Merge gap: 0.9s | Regions: 2
  0.02-22.75s (22.73s)
  23.82-31.86s (8.04s)

Merge gap: 1.0s | Regions: 2
  0.02-22.75s (22.73s)
  23.82-31.86s (8.04s)


In [130]:
import inspect
import call_analysis

print(inspect.signature(call_analysis.build_analysis_windows))
print(inspect.getsource(call_analysis.build_analysis_windows))

(regions: 'List[SpeechRegion]', sample_rate: 'int', window_seconds: 'float' = 6.0, hop_seconds: 'float' = 3.0, min_trailing_seconds: 'float' = 2.0) -> 'List[AnalysisWindow]'
def build_analysis_windows(
    regions: List[SpeechRegion],
    sample_rate: int,
    window_seconds: float = 6.0,
    hop_seconds: float = 3.0,
    min_trailing_seconds: float = 2.0,
) -> List[AnalysisWindow]:
    """Create overlapping analysis windows within speech regions."""
    if sample_rate <= 0:
        raise ValueError("sample_rate must be positive.")
    if window_seconds <= 0 or hop_seconds <= 0 or min_trailing_seconds < 0:
        raise ValueError("Window, hop, and trailing durations must be valid.")

    windows = []
    for region in regions:
        if region.start_seconds < 0 or region.end_seconds < region.start_seconds:
            raise ValueError("Invalid speech region.")

        start = region.start_seconds
        while start < region.end_seconds:
            duration = min(window_seconds, re

In [131]:
from call_analysis import detect_speech_regions, build_analysis_windows

# Experimental segmentation
test_regions = detect_speech_regions(
    waveform,
    sample_rate=16000,
    merge_gap_seconds=0.5,
    min_region_seconds=0.15,
)

# Build windows using the existing production settings
test_windows = build_analysis_windows(
    test_regions,
    sample_rate=16000,
    window_seconds=6.0,
    hop_seconds=3.0,
    min_trailing_seconds=2.0,
)

print("Experimental window scores (0.5s gap merging):\n")

for window in test_windows:
    clip = waveform[window.start_sample:window.end_sample]

    raw_logit, p_fake = mp.score_waveform(
        clip,
        sample_rate=16000,
        bundle=bundle,
    )

    verdict = mp.verdict_from_p_fake(p_fake)

    print(
        f"{window.start_seconds:.2f}-{window.end_seconds:.2f}s | "
        f"Duration: {len(clip)/16000:.2f}s | "
        f"P(fake): {p_fake:.4f} | "
        f"Verdict: {verdict}"
    )

Experimental window scores (0.5s gap merging):

1.91-7.91s | Duration: 6.00s | P(fake): 0.0020 | Verdict: real
4.91-10.91s | Duration: 6.00s | P(fake): 0.0070 | Verdict: real
7.91-11.36s | Duration: 3.44s | P(fake): 0.0023 | Verdict: real
11.87-14.87s | Duration: 3.00s | P(fake): 0.1227 | Verdict: real
17.03-22.75s | Duration: 5.71s | P(fake): 0.0123 | Verdict: real
20.03-22.75s | Duration: 2.71s | P(fake): 0.3182 | Verdict: real
24.95-30.95s | Duration: 6.00s | P(fake): 0.1001 | Verdict: real
27.95-31.86s | Duration: 3.92s | P(fake): 0.0121 | Verdict: real


In [132]:
%cd /content/indic-fake-voice-detection
!pytest -q

/content/indic-fake-voice-detection
..............................                                           [100%]
30 passed in 3.56s


In [133]:
%cd /content/indic-fake-voice-detection
!grep -nE '^def test_|^    def test_' tests/test_call_analysis.py

/content/indic-fake-voice-detection
15:def test_silence_produces_no_speech_regions():
23:def test_quiet_speech_is_detected():
32:def test_silence_gap_creates_separate_speech_regions():
45:def test_windows_use_six_seconds_and_three_second_hop():
56:def test_trailing_window_shorter_than_two_seconds_is_dropped():
75:def test_call_and_single_clip_upload_policies_remain_separate():
83:def test_recording_over_fifteen_minutes_is_rejected(monkeypatch):


In [134]:
%cd /content/indic-fake-voice-detection
!grep -RniE '250|200.?1024|progress_callback|flagged_ranges|all.real|mixed|fifteen|900|duration' tests/

/content/indic-fake-voice-detection
tests/test_voice_guard_core.py:42:    assert metrics["duration_seconds"] == 2.0
tests/test_call_analysis.py:10:def make_tone(duration, sample_rate=16000, amplitude=0.08):
tests/test_call_analysis.py:11:    time = np.arange(int(duration * sample_rate), dtype=np.float32) / sample_rate
tests/test_call_analysis.py:77:    assert CallInputPolicy().max_duration_seconds == 900.0
tests/test_call_analysis.py:80:    assert AudioInputPolicy().max_duration_seconds == 30.0
tests/test_call_analysis.py:83:def test_recording_over_fifteen_minutes_is_rejected(monkeypatch):
tests/test_evidence_report.py:16:            "duration_seconds": 4.2,
grep: tests/__pycache__/test_voice_guard_core.cpython-313-pytest-8.4.2.pyc: binary file matches
grep: tests/__pycache__/test_evidence_report.cpython-313-pytest-8.4.2.pyc: binary file matches
grep: tests/__pycache__/test_call_analysis.cpython-313-pytest-8.4.2.pyc: binary file matches


In [135]:
%cd /content/indic-fake-voice-detection
!sed -n '1,260p' call_analysis.py

/content/indic-fake-voice-detection
"""Long-call speech detection and window segmentation for Voice Guard."""

from __future__ import annotations

from dataclasses import dataclass
from typing import List

import numpy as np


@dataclass(frozen=True)
class SpeechRegion:
    """Speech-active interval in the original recording timeline."""

    start_seconds: float
    end_seconds: float


@dataclass(frozen=True)
class AnalysisWindow:
    """A model-analysis window with its original recording timestamps."""

    start_seconds: float
    end_seconds: float
    start_sample: int
    end_sample: int


def detect_speech_regions(
    waveform: np.ndarray,
    sample_rate: int,
    frame_duration: float = 0.025,
    hop_duration: float = 0.010,
    merge_gap_seconds: float = 0.30,
    min_region_seconds: float = 0.15,
    energy_threshold: float | None = None,
) -> List[SpeechRegion]:
    """Find speech-active regions using a lightweight adaptive energy gate.

    This is an energy-based detec

In [136]:
%cd /content/indic-fake-voice-detection
!sed -n '220,380p' call_analysis.py

/content/indic-fake-voice-detection
    import model_pipeline

    from voice_guard_core import (
        CallInputPolicy,
        validate_encoded_audio_size,
    )

    validate_encoded_audio_size(
        audio_bytes,
        policy=CallInputPolicy(),
    )

    waveform, original_sample_rate = model_pipeline.decode_audio_bytes(
        audio_bytes
    )

    waveform = np.asarray(waveform, dtype=np.float32).reshape(-1)

    if waveform.size == 0:
        raise ValueError("The decoded recording is empty.")
    if original_sample_rate <= 0:
        raise ValueError("Invalid audio sample rate.")
    if not np.isfinite(waveform).all():
        raise ValueError("The recording contains non-finite audio values.")

    duration_seconds = waveform.size / float(original_sample_rate)

    if duration_seconds > 900.0:
        raise ValueError("Call recordings cannot exceed 15 minutes.")
    if duration_seconds < 1.0:
        raise ValueError("The recording must be at least 1 second long.")

  

In [137]:
%cd /content/indic-fake-voice-detection
!cat tests/test_call_analysis.py

/content/indic-fake-voice-detection
import numpy as np

from call_analysis import (
    SpeechRegion,
    build_analysis_windows,
    detect_speech_regions,
)


def make_tone(duration, sample_rate=16000, amplitude=0.08):
    time = np.arange(int(duration * sample_rate), dtype=np.float32) / sample_rate
    return amplitude * np.sin(2 * np.pi * 220 * time)


def test_silence_produces_no_speech_regions():
    silence = np.zeros(5 * 16000, dtype=np.float32)

    regions = detect_speech_regions(silence, 16000)

    assert regions == []


def test_quiet_speech_is_detected():
    quiet_speech = make_tone(2.0, amplitude=0.003)

    regions = detect_speech_regions(quiet_speech, 16000)

    assert len(regions) == 1
    assert regions[0].end_seconds - regions[0].start_seconds >= 1.8


def test_silence_gap_creates_separate_speech_regions():
    first = make_tone(2.0)
    silence = np.zeros(int(0.8 * 16000), dtype=np.float32)
    second = make_tone(2.0)
    waveform = np.concatenate([first, silence

In [138]:
%cd /content/indic-fake-voice-detection
!grep -n -A35 -B5 'def validate_encoded_audio_size' voice_guard_core.py

/content/indic-fake-voice-detection
78-
79-DEFAULT_AUDIO_POLICY = AudioInputPolicy()
80-DEFAULT_DECISION_POLICY = DecisionPolicy()
81-
82-
83:def validate_encoded_audio_size(
84-    audio_bytes: bytes,
85-    policy: AudioInputPolicy = DEFAULT_AUDIO_POLICY,
86-) -> None:
87-    if not isinstance(audio_bytes, (bytes, bytearray)):
88-        raise AudioValidationError("The uploaded audio could not be read.")
89-    if not audio_bytes:
90-        raise AudioValidationError("The uploaded audio file is empty.")
91-    if len(audio_bytes) > policy.max_bytes:
92-        limit_mb = policy.max_bytes / (1024 * 1024)
93-        raise AudioValidationError(
94-            f"The audio file is too large. Upload a file no larger than {limit_mb:.0f} MB."
95-        )
96-
97-
98-def validate_decoded_audio(
99-    waveform: np.ndarray,
100-    sample_rate: int,
101-    policy: AudioInputPolicy = DEFAULT_AUDIO_POLICY,
102-) -> Dict[str, Any]:
103-    """Validate decoded mono audio and return transparent q

In [139]:
%cd /content/indic-fake-voice-detection
!sed -n '1,65p' call_analysis.py

/content/indic-fake-voice-detection
"""Long-call speech detection and window segmentation for Voice Guard."""

from __future__ import annotations

from dataclasses import dataclass
from typing import List

import numpy as np


@dataclass(frozen=True)
class SpeechRegion:
    """Speech-active interval in the original recording timeline."""

    start_seconds: float
    end_seconds: float


@dataclass(frozen=True)
class AnalysisWindow:
    """A model-analysis window with its original recording timestamps."""

    start_seconds: float
    end_seconds: float
    start_sample: int
    end_sample: int


def detect_speech_regions(
    waveform: np.ndarray,
    sample_rate: int,
    frame_duration: float = 0.025,
    hop_duration: float = 0.010,
    merge_gap_seconds: float = 0.30,
    min_region_seconds: float = 0.15,
    energy_threshold: float | None = None,
) -> List[SpeechRegion]:
    """Find speech-active regions using a lightweight adaptive energy gate.

    This is an energy-based detec

In [140]:
%cd /content/indic-fake-voice-detection
!grep -n -A55 -B5 'class CallAnalysisReport\|def analyze_call' call_analysis.py

/content/indic-fake-voice-detection
163-    p_fake: float
164-    verdict: str
165-
166-
167-@dataclass(frozen=True)
168:class CallAnalysisReport:
169-    """Complete analysis results for one phone-call recording."""
170-
171-    duration_seconds: float
172-    speech_duration_seconds: float
173-    windows_analysed: int
174-    flagged_windows: int
175-    flagged_share: float
176-    highest_p_fake: float
177-    highest_score_timestamp: float
178-    window_scores: List[WindowScore]
179-    flagged_ranges: List[SpeechRegion]
180-
181-
182-def merge_flagged_ranges(
183-    window_scores: List[WindowScore],
184-) -> List[SpeechRegion]:
185-    """Merge overlapping or adjacent windows classified as fake."""
186-    flagged = sorted(
187-        (score for score in window_scores if score.verdict == "fake"),
188-        key=lambda score: score.start_seconds,
189-    )
190-
191-    if not flagged:
192-        return []
193-
194-    merged = []
195-    current_start = flagged[0].start_seco

In [141]:
%cd /content/indic-fake-voice-detection
!sed -n '259,340p' call_analysis.py

/content/indic-fake-voice-detection
    regions = detect_speech_regions(waveform, sample_rate)
    windows = build_analysis_windows(
        regions,
        sample_rate,
        window_seconds=window_seconds,
        hop_seconds=hop_seconds,
    )

    if not windows:
        raise ValueError(
            "No speech-active regions long enough for analysis were detected."
        )

    if checkpoint_dir is None:
        bundle = model_pipeline.load_inference_bundle()
    else:
        bundle = model_pipeline.load_inference_bundle(
            checkpoint_dir=checkpoint_dir
        )

    window_scores = []

    for index, window in enumerate(windows):
        segment = waveform[window.start_sample:window.end_sample]

        raw_logit, p_fake = model_pipeline.score_waveform(
            segment,
            sample_rate,
            bundle=bundle,
        )
        verdict = model_pipeline.verdict_from_p_fake(p_fake)

        window_scores.append(
            WindowScore(
              

In [143]:
%%writefile /content/indic-fake-voice-detection/tests/test_call_analysis.py
import numpy as np
import pytest

from call_analysis import (
    SpeechRegion,
    analyze_call,
    build_analysis_windows,
    detect_speech_regions,
)
from voice_guard_core import (
    AudioInputPolicy,
    CallInputPolicy,
    validate_encoded_audio_size,
)


def make_tone(duration, sample_rate=16000, amplitude=0.08):
    time = np.arange(int(duration * sample_rate), dtype=np.float32) / sample_rate
    return amplitude * np.sin(2 * np.pi * 220 * time)


def make_two_section_recording():
    first = make_tone(6.0)
    silence = np.zeros(int(0.8 * 16000), dtype=np.float32)
    second = make_tone(6.0)
    return np.concatenate([first, silence, second])


def patch_call_pipeline(monkeypatch, waveform, probabilities):
    import model_pipeline

    monkeypatch.setattr(
        model_pipeline, "decode_audio_bytes",
        lambda audio_bytes: (waveform, 16000),
    )
    monkeypatch.setattr(
        model_pipeline, "ensure_target_sample_rate",
        lambda samples, original_rate, target_rate: samples,
    )
    monkeypatch.setattr(
        model_pipeline, "load_inference_bundle",
        lambda **kwargs: object(),
    )

    scores = iter(probabilities)
    monkeypatch.setattr(
        model_pipeline, "score_waveform",
        lambda segment, sample_rate, bundle=None: (0.0, next(scores)),
    )


def test_silence_produces_no_speech_regions():
    silence = np.zeros(5 * 16000, dtype=np.float32)
    assert detect_speech_regions(silence, 16000) == []


def test_quiet_speech_is_detected():
    quiet_speech = make_tone(2.0, amplitude=0.003)
    regions = detect_speech_regions(quiet_speech, 16000)
    assert len(regions) == 1
    assert regions[0].end_seconds - regions[0].start_seconds >= 1.8


def test_silence_gap_creates_separate_speech_regions():
    waveform = np.concatenate([
        make_tone(2.0),
        np.zeros(int(0.8 * 16000), dtype=np.float32),
        make_tone(2.0),
    ])
    regions = detect_speech_regions(waveform, 16000)
    assert len(regions) == 2
    assert regions[0].end_seconds < regions[1].start_seconds
    assert 0.5 <= regions[1].start_seconds - regions[0].end_seconds <= 1.1


def test_windows_use_six_seconds_and_three_second_hop():
    windows = build_analysis_windows([SpeechRegion(0.0, 13.0)], 16000)
    assert [window.start_seconds for window in windows] == [0.0, 3.0, 6.0, 9.0]
    assert windows[0].end_seconds == 6.0
    assert windows[1].end_seconds == 9.0


def test_trailing_window_shorter_than_two_seconds_is_dropped():
    windows = build_analysis_windows([SpeechRegion(0.0, 7.0)], 16000)
    assert all(window.end_seconds - window.start_seconds >= 2.0 for window in windows)
    assert len(windows) == 2
    assert windows[-1].start_seconds == 3.0
    assert windows[-1].end_seconds == 7.0


def test_call_and_single_clip_upload_policies_remain_separate():
    assert CallInputPolicy().max_bytes == 200 * 1024 * 1024
    assert CallInputPolicy().max_duration_seconds == 900.0
    assert AudioInputPolicy().max_bytes == 25 * 1024 * 1024
    assert AudioInputPolicy().max_duration_seconds == 30.0


def test_call_upload_size_limit_is_enforced():
    from dataclasses import replace
    from voice_guard_core import AudioValidationError

    small_policy = replace(CallInputPolicy(), max_bytes=10)
    with pytest.raises(AudioValidationError, match="too large"):
        validate_encoded_audio_size(b"x" * 11, policy=small_policy)


def test_recording_over_fifteen_minutes_is_rejected(monkeypatch):
    import model_pipeline

    monkeypatch.setattr(
        model_pipeline, "decode_audio_bytes",
        lambda audio_bytes: (np.zeros(901, dtype=np.float32), 1),
    )

    def unexpected_resampling(*args, **kwargs):
        raise AssertionError("Resampling should not run for an overlong call.")

    monkeypatch.setattr(
        model_pipeline, "ensure_target_sample_rate",
        unexpected_resampling,
    )

    with pytest.raises(ValueError, match="15 minutes"):
        analyze_call(b"test-audio")


def test_mixed_call_reports_flagged_range_and_progress(monkeypatch):
    waveform = make_two_section_recording()
    patch_call_pipeline(monkeypatch, waveform, [0.1, 0.8])

    progress = []
    report = analyze_call(b"test-audio", progress_callback=progress.append)

    assert report.windows_analysed == 2
    assert report.flagged_windows == 1
    assert report.flagged_share == pytest.approx(0.5)
    assert len(report.flagged_ranges) == 1
    assert report.flagged_ranges[0].start_seconds > 6.5
    assert report.flagged_ranges[0].end_seconds > report.flagged_ranges[0].start_seconds
    assert report.highest_p_fake == pytest.approx(0.8)
    assert report.highest_score_timestamp == report.window_scores[1].start_seconds
    assert progress == pytest.approx([0.5, 1.0])


def test_all_real_call_has_no_flagged_ranges(monkeypatch):
    waveform = make_two_section_recording()
    patch_call_pipeline(monkeypatch, waveform, [0.1, 0.2])

    report = analyze_call(b"test-audio")

    assert report.windows_analysed == 2
    assert report.flagged_windows == 0
    assert report.flagged_share == 0.0
    assert report.flagged_ranges == []

Overwriting /content/indic-fake-voice-detection/tests/test_call_analysis.py


In [145]:
%cd /content/indic-fake-voice-detection

import numpy as np
from call_analysis import detect_speech_regions, build_analysis_windows

first = 0.08 * np.sin(2 * np.pi * 220 * np.arange(6 * 16000) / 16000)
silence = np.zeros(int(0.8 * 16000), dtype=np.float32)
second = 0.08 * np.sin(2 * np.pi * 220 * np.arange(6 * 16000) / 16000)
waveform = np.concatenate([first, silence, second]).astype(np.float32)

regions = detect_speech_regions(waveform, 16000)
windows = build_analysis_windows(regions, 16000)

print("Speech regions:", regions)
print("Analysis windows:", len(windows))
print("Window timestamps:", [(w.start_seconds, w.end_seconds) for w in windows])

/content/indic-fake-voice-detection
Speech regions: [SpeechRegion(start_seconds=0.0, end_seconds=6.015), SpeechRegion(start_seconds=6.78, end_seconds=12.795)]
Analysis windows: 4
Window timestamps: [(0.0, 6.0), (3.0, 6.015), (6.78, 12.780000000000001), (9.780000000000001, 12.795)]


In [146]:
%cd /content/indic-fake-voice-detection

from pathlib import Path

path = Path("tests/test_call_analysis.py")
content = path.read_text()

content = content.replace(
    "patch_call_pipeline(monkeypatch, waveform, [0.1, 0.8])",
    "patch_call_pipeline(monkeypatch, waveform, [0.1, 0.1, 0.8, 0.8])",
)
content = content.replace(
    "patch_call_pipeline(monkeypatch, waveform, [0.1, 0.2])",
    "patch_call_pipeline(monkeypatch, waveform, [0.1, 0.1, 0.2, 0.2])",
)

path.write_text(content)
print("Updated both test cases with four scores.")

/content/indic-fake-voice-detection
Updated both test cases with four scores.


In [148]:
%cd /content/indic-fake-voice-detection

from pathlib import Path

path = Path("tests/test_call_analysis.py")
content = path.read_text()

content = content.replace(
    "assert report.windows_analysed == 2",
    "assert report.windows_analysed == 4",
)
content = content.replace(
    "assert report.flagged_windows == 1",
    "assert report.flagged_windows == 2",
)
content = content.replace(
    "assert report.flagged_share == pytest.approx(0.5)",
    "assert report.flagged_share == pytest.approx(0.5)",
)
content = content.replace(
    "assert progress == pytest.approx([0.5, 1.0])",
    "assert progress == pytest.approx([0.25, 0.5, 0.75, 1.0])",
)

path.write_text(content)
print("Updated window counts and progress assertions.")

/content/indic-fake-voice-detection
Updated window counts and progress assertions.


In [150]:
%cd /content/indic-fake-voice-detection

from pathlib import Path

path = Path("tests/test_call_analysis.py")
content = path.read_text().replace(
    "assert report.highest_score_timestamp == report.window_scores[1].start_seconds",
    "assert report.highest_score_timestamp == report.window_scores[2].start_seconds",
)
path.write_text(content)

print("Timestamp assertion corrected.")

/content/indic-fake-voice-detection
Timestamp assertion corrected.


In [151]:
!pytest -q

.................................                                        [100%]
33 passed in 2.40s


In [152]:
%cd /content/indic-fake-voice-detection
!git status --short --branch
!git diff --check
!git diff --stat

/content/indic-fake-voice-detection
## feature/long-call-analysis
 M voice_guard_core.py
?? call_analysis.py
?? tests/test_call_analysis.py
 voice_guard_core.py | 9 +++++++++
 1 file changed, 9 insertions(+)


In [153]:
%cd /content/indic-fake-voice-detection
!git diff -- voice_guard_core.py
!python -m py_compile call_analysis.py voice_guard_core.py tests/test_call_analysis.py
!wc -l call_analysis.py tests/test_call_analysis.py

/content/indic-fake-voice-detection
diff --git a/voice_guard_core.py b/voice_guard_core.py
index 39b83e8..1df28d8 100644
--- a/voice_guard_core.py
+++ b/voice_guard_core.py
@@ -50,6 +50,15 @@ class AudioInputPolicy:
     min_active_seconds: float = 0.45
 
 
+@dataclass(frozen=True)
+class CallInputPolicy:
+    """Input limits for long phone-call recordings."""
+
+    max_bytes: int = 200 * 1024 * 1024
+    min_duration_seconds: float = 1.0
+    max_duration_seconds: float = 900.0
+
+
 @dataclass(frozen=True)
 class DecisionPolicy:
     """Three-way abstention policy operating on calibrated P(fake)."""
  327 call_analysis.py
  154 tests/test_call_analysis.py
  481 total


In [155]:
%cd /content/indic-fake-voice-detection
!git add voice_guard_core.py call_analysis.py tests/test_call_analysis.py && git commit -m "Add long-call analysis core"

/content/indic-fake-voice-detection
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@2e6e7fcccb3b.(none)')


In [156]:
%cd /content/indic-fake-voice-detection
!git config user.name "Ashitha098"
!git config user.email "rajashitha310@gmail.com"

/content/indic-fake-voice-detection


In [157]:
%cd /content/indic-fake-voice-detection
!git config user.name "Ashitha098"
!git config user.email "rajashitha310@gmail.com"

/content/indic-fake-voice-detection


In [158]:
%cd /content/indic-fake-voice-detection
!git config user.name "Ashitha098"
!git config user.email "rajashitha310@gmail.com"

/content/indic-fake-voice-detection


In [159]:
%cd /content/indic-fake-voice-detection
!git config user.name
!git config user.email

/content/indic-fake-voice-detection
Ashitha098
rajashitha310@gmail.com


In [160]:
%cd /content/indic-fake-voice-detection
!git add voice_guard_core.py call_analysis.py tests/test_call_analysis.py && git commit -m "Add long-call analysis core"

/content/indic-fake-voice-detection
[feature/long-call-analysis e1fca64] Add long-call analysis core
 3 files changed, 490 insertions(+)
 create mode 100644 call_analysis.py
 create mode 100644 tests/test_call_analysis.py


In [161]:
%cd /content/indic-fake-voice-detection
!git status --short --branch
!git log -1 --oneline

/content/indic-fake-voice-detection
## feature/long-call-analysis
e1fca64 (HEAD -> feature/long-call-analysis) Add long-call analysis core
